# Pengindeksan Bahaya Seismik Indonesia dengan Discrete Global Grid System (H3-SHI)

**Proyek Akhir — Teknologi Big Data | Kelompok 8**

Implementasi PRD v2.1. Pengindeksan bahaya seismik di atas Uber H3 Discrete Global
Grid System Resolusi 5, menggabungkan estimasi nilai *b* Gutenberg-Richter dengan
Seismic Hazard Index multi-kriteria yang divalidasi dengan pembobotan entropi.

## Rumusan masalah

Algoritma clustering konvensional memaksakan geometri yang tidak dimiliki gempa bumi.
$k$-means memartisi ruang menjadi sel Voronoi di sekitar centroid, yang bentuknya
konveks dan kurang lebih isotropik. DBSCAN dengan metrik Euclidean menumbuhkan cluster
ke luar pada radius seragam $\varepsilon$. Seismisitas Indonesia tidak konveks dan tidak
isotropik: ia mengikuti megathrust Sunda sebagai jalur kurvilinear sepanjang sekitar
4.000 km, Busur Banda sebagai tapal kuda yang rapat, serta sistem Sesar Sumatra dan
Palu-Koro sebagai lineamen strike-slip yang sempit. Memaksakan cluster isotropik ke
fitur linear berujung pada dua kegagalan: satu sesar terpecah ke banyak cluster, atau
satu sesar melebur dengan seismisitas tak berhubungan di sebelahnya.

Discrete Global Grid System menghindari masalah ini dengan tidak menyimpulkan bentuk
sama sekali. Gridnya ditetapkan di awal, setiap sel punya luas dan topologi tetangga
yang sama, dan struktur spasial bahaya muncul dari statistik per sel, bukan dari prior
geometris sebuah algoritma.

**Geometri sel.** H3 Resolusi 5 memberi sel seluas rata-rata **252,9 km²** dengan panjang
sisi sekitar **8,5 km**. Angka ini perlu ditegaskan, karena PRD menyebut "≈1.570 km²,
sisi ≈25 km" untuk Resolusi 5 dan itu keliru: luas sebesar itu milik **Resolusi 4**
(1.770,3 km², sisi ≈22,6 km). Naik satu resolusi membagi luas sel kira-kira tujuh, jadi
res 4 → 1.770 km², res 5 → 252,9 km², res 6 → 36,1 km².

Konsekuensinya bukan cuma salah angka. Alasan pemilihan resolusi yang ditulis di PRD,
yaitu "skala di mana estimasi nilai *b* punya cukup event per sel", justru menunjuk ke
Resolusi 4, bukan 5. Tabel sensitivitas di Bagian 7 memberi datanya: di res 4 ada 96%
sel dengan nilai *b* terukur, di res 5 tinggal 76%, dan di res 6 hanya 31%. Resolusi 5
dipertahankan sebagai baseline supaya sebanding dengan run sebelumnya, tetapi laporan
harus menyatakan pilihan ini secara sadar dan mempertimbangkan res 4 sebagai angka utama.

## Apa indeks ini, dan apa yang bukan

SHI adalah peringkat bahaya relatif yang diturunkan dari katalog, pada skala absolut
0–1. Tiga dari empat kanalnya dinormalisasi terhadap domain rujukan fisis yang tetap,
sehingga nilainya tetap sebanding antar-run dan antar-resolusi.

SHI bukan probabilistic seismic hazard assessment. Tidak ada periode ulang, tidak ada
model ground motion, tidak ada respons situs, tidak ada lapisan keterpaparan atau
kerentanan. Yang dilakukannya adalah memeringkat sel grid berdasarkan proksi bahaya
yang teramati di katalog. Perbandingan terhadap peta PusGen adalah perbandingan pola
spasial, bukan validasi numerik.

## Alur pipeline

| Bagian | Tahap |
| :-- | :-- |
| 1 | Lingkungan kerja dan ingesti data |
| 2 | Audit kualitas data dan penyaringan |
| 3 | Binning spasial H3 |
| 4 | Magnitudo kelengkapan $M_c$ |
| 5 | Pipeline SHI: nilai *b*, normalisasi, pembobotan |
| 6 | Assertion validasi |
| 7 | Analisis sensitivitas |
| 8 | Visualisasi statis |
| 9 | Ekspor GeoJSON dan spatial join administratif |
| 10 | Choropleth interaktif |
| 11 | Ringkasan deliverable |

---
## 1. Lingkungan Kerja dan Ingesti Data

Pin `h3>=4` itu penting. API v3 memakai `geo_to_h3` / `k_ring` / `h3_to_geo_boundary`,
sedangkan seluruh pemanggilan di sini memakai nama v4. Mencampur versi memunculkan
`AttributeError`, bukan diam-diam menghasilkan angka yang salah, jadi assertion di bawah
gagal lebih awal.

### Cara menjalankan

**Di Colab,** yang paling sedikit hambatannya. Lepas komentar baris `pip install`,
jalankan sel mount Drive dan setujui izinnya, lalu Runtime → Run all. Katalognya dicari
sendiri di `MyDrive` dengan pencocokan pola, jadi letak foldernya tidak perlu diketahui.

**Secara lokal,** letakkan CSV-nya di folder yang sama dengan notebook ini. Paket yang
dibutuhkan harus sudah terpasang lebih dulu; `geopandas` dan `h3` di Windows lebih
mudah dipasang lewat conda daripada pip.

Nama file tidak perlu persis. Pencarian memakai pola
`Integrated_Standardized_Earthquake_Catalogue*.csv`, sehingga salinan bersuffix
seperti `... (1).csv` tetap terbaca.

In [ ]:
# Colab: lepas komentar pada run pertama
# !pip install -q "h3>=4.0" pandas numpy scipy geopandas folium matplotlib seaborn branca shapely

In [ ]:
# Mount Google Drive kalau berjalan di Colab, supaya dataset di MyDrive bisa dibaca.
# Di lingkungan lokal blok ini tidak melakukan apa pun.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Bukan Colab: dataset akan dicari di folder lokal.")

In [ ]:
import os
import time

import h3
import folium
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import matplotlib.pyplot as plt
import seaborn as sns

# Yang diuji adalah pemanggilan yang benar-benar kita pakai, bukan string versi
assert hasattr(h3, "latlng_to_cell"), "Butuh API h3 v4: pip install 'h3>=4'"

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 50)

print(f"h3 {h3.__version__} | pandas {pd.__version__} | geopandas {gpd.__version__}")

In [ ]:
import glob

# Dicari dengan pola, bukan nama persis: unduhan dari Drive sering menambahkan " (1)",
# dan di Colab letak foldernya di dalam MyDrive tidak diketahui sebelumnya.
PATTERN = "Integrated_Standardized_Earthquake_Catalogue*.csv"
SEARCH_DIRS = [".", "/content", "/content/drive/MyDrive", "/content/drive/My Drive"]

hits = []
for base in SEARCH_DIRS:
    if os.path.isdir(base):
        hits += glob.glob(os.path.join(base, "**", PATTERN), recursive=True)

assert hits, (
    f"Katalog tidak ditemukan. Dicari pola {PATTERN!r} di: {SEARCH_DIRS}.\n"
    f"Di Colab: mount Drive dulu (lihat sel Markdown di atas). "
    f"Secara lokal: letakkan CSV-nya di samping notebook ini, "
    f"atau set CSV_PATH secara manual."
)
CSV_PATH = sorted(hits, key=len)[0]      # nama terpendek = tanpa suffix " (1)"

if len(hits) > 1:
    print(f"Ada {len(hits)} salinan katalog, dipakai yang ini:")
    for h in sorted(hits, key=len):
        print(f"  {'->' if h == CSV_PATH else '  '} {h}")
print(f"Katalog: {CSV_PATH}")
print(f"Ukuran:  {os.path.getsize(CSV_PATH) / 1e6:.1f} MB")

In [ ]:
# Batas administratif GADM level 1, opsional, dipakai Bagian 9. Diresolusi di sini juga
# supaya sudah ketahuan sejak awal apakah spatial join akan jalan, bukan baru ketemu
# masalahnya setelah sepuluh menit pipeline.
#
# Filter os.path.isfile() itu wajib, bukan hiasan: mengekstrak unduhan GADM sering
# menghasilkan folder bernama "gadm41_IDN_1.json" yang berisi file dengan nama sama.
# Pola glob mencocoki folder itu juga, dan gpd.read_file() akan crash di atas folder.
ADM_PATTERNS = ["gadm41_IDN_1.json", "gadm41_IDN_1.geojson", "gadm41_IDN_1.shp"]

adm_hits = []
for base in SEARCH_DIRS:
    if os.path.isdir(base):
        for pat in ADM_PATTERNS:
            adm_hits += [p for p in glob.glob(os.path.join(base, "**", pat), recursive=True)
                         if os.path.isfile(p)]

ADM_PATH = sorted(adm_hits, key=len)[0] if adm_hits else None

if ADM_PATH:
    print(f"Batas administratif: {ADM_PATH}")
    print(f"Ukuran:              {os.path.getsize(ADM_PATH) / 1e6:.1f} MB")
else:
    print("Batas administratif tidak ditemukan. Bagian 9 akan melewati spatial join,")
    print("dan deliverable 5.1.11 (profil provinsi Top-20) belum lengkap.")
    print(f"Dicari pola {ADM_PATTERNS} di: {SEARCH_DIRS}")
    print("\nUnduh GADM level 1 Indonesia dari https://gadm.org/download_country.html")
    print("lalu taruh di MyDrive (Colab) atau di samping notebook ini (lokal).")

In [ ]:
# utf-8-sig membuang BOM di awal file, yang kalau tidak akan membuat kolom pertama
# bernama "﻿No". low_memory=False menghindari DtypeWarning dari kolom Remarks
# yang tipenya campur, sesuai checklist item 1 (nol warning).
df_raw = pd.read_csv(CSV_PATH, encoding="utf-8-sig", low_memory=False)
print(f"Jumlah record mentah: {len(df_raw):,}")
print(f"Kolom asli: {list(df_raw.columns)}")

# Katalog ini memakai kapitalisasi dan nama yang berbeda dari yang dipakai pipeline.
# Dipetakan sekali di sini, sehingga sisa notebook bekerja dengan satu konvensi saja.
RENAME = {
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Depth_km": "depth",
    "Magnitude": "magnitude",
}
applied = {k: v for k, v in RENAME.items() if k in df_raw.columns}
df_raw = df_raw.rename(columns=applied)
if applied:
    print(f"\nKolom dipetakan: {applied}")

In [ ]:
REQUIRED = ["latitude", "longitude", "depth", "magnitude"]
missing = [c for c in REQUIRED if c not in df_raw.columns]
assert not missing, (
    f"Katalog kehilangan kolom wajib: {missing}. "
    f"Kolom yang tersedia: {list(df_raw.columns)}. Tambahkan pemetaannya ke RENAME."
)

# Kolom tahun. Katalog ini punya "Year" sebagai integer bersih, sementara "Date" untuk
# event kuno berbentuk placeholder seperti "00-000-0088" yang tidak bisa diparse.
# Memakai "Year" langsung menghindari kekacauan itu.
if "Year" in df_raw.columns:
    YEAR_SOURCE = "Year"
else:
    YEAR_SOURCE = next(c for c in ("time", "Date", "date", "datetime")
                       if c in df_raw.columns)
print(f"Sumber tahun: {YEAR_SOURCE}")
print("Pemeriksaan skema lolos.")

In [ ]:
# Folder tujuan semua berkas keluaran.
#
# Di Colab, path relatif jatuh ke /content, yaitu folder sesi yang DIHAPUS begitu
# runtime putus. Artinya peta, GeoJSON, CSV dan gambar semuanya lenyap kecuali sempat
# diunduh manual. Karena itu di Colab semuanya ditulis ke Drive supaya ikut tersimpan.
OUT_DIR = "/content/drive/MyDrive/h3shi_output" if IN_COLAB else "."
os.makedirs(OUT_DIR, exist_ok=True)

def out(nama):
    """Path lengkap untuk satu berkas keluaran."""
    return os.path.join(OUT_DIR, nama)

print(f"Folder keluaran: {os.path.abspath(OUT_DIR)}")
print("Tersimpan di Drive, jadi tetap ada setelah sesi Colab berakhir."
      if IN_COLAB else "Berjalan lokal: berkas ditulis di samping notebook ini.")

---
## 2. Audit Kualitas Data dan Penyaringan

Empat filter, masing-masing dengan alasannya.

1. **Temporal, $\ge 1960$.** Katalog ini mundur sampai 88 M, tetapi kelengkapan era
   prainstrumental sama sekali tidak seragam. Mencampur kedua era akan membuat jumlah
   event menjadi catatan historiografi, bukan catatan seismisitas.
2. **Kotak pembatas spasial,** lintang $[-11, 6]$ dan bujur $[95, 141]$. Ini kotak,
   bukan garis pantai: di dalamnya ada laut lepas dan serpihan wilayah negara tetangga.
   Untuk studi tektonik hal itu memang disengaja, dan justru karena itu Bagian 9 harus
   menangani sel lepas pantai secara eksplisit pada spatial join administratif.
3. **Nilai hilang** pada salah satu dari empat field numerik.
4. **Rentang kedalaman** $[0, 700]$ km, menolak kedalaman negatif dan nilai yang lebih
   dalam dari hiposenter terdalam yang pernah tercatat.

Audit ini juga melaporkan dua hal yang mengubah cara hasilnya dibaca: proporsi event
yang membawa kedalaman default 10 km, dan apakah katalog mencampur skala magnitudo.

In [ ]:
df = df_raw.copy()
stages, counts = ["mentah"], [len(df)]

# 1. Era instrumental
if YEAR_SOURCE == "Year":
    df["year"] = pd.to_numeric(df["Year"], errors="coerce")
else:
    df["year"] = pd.to_datetime(df[YEAR_SOURCE], errors="coerce").dt.year
df = df[df["year"] >= 1960]
stages.append(">= 1960"); counts.append(len(df))

# 2. Kotak pembatas tektonik Indonesia
df = df[df["latitude"].between(-11.0, 6.0) & df["longitude"].between(95.0, 141.0)]
stages.append("dalam bbox"); counts.append(len(df))

# 3. Record numerik lengkap
df = df.dropna(subset=REQUIRED).copy()
stages.append("baris lengkap"); counts.append(len(df))

# 4. Kedalaman yang mungkin secara fisis
df = df[df["depth"].between(0.0, 700.0)].copy()
stages.append("depth dalam [0,700]"); counts.append(len(df))

audit = pd.DataFrame({"tahap": stages, "record": counts})
audit["terbuang"] = -audit["record"].diff().fillna(0).astype(int)
audit["sisa_persen"] = (audit["record"] / counts[0] * 100).round(2)
print(audit.to_string(index=False))

In [ ]:
# Audit kedalaman default. Katalog memberi kedalaman tetap ketika inversi kedalaman
# gagal, dan lonjakan di nilai-nilai itu berarti kanal shallow-ratio ikut mengukur data
# hilang. Katalog ini punya lebih dari satu: 10 km (default modern) dan 35 km, sementara
# 33 km adalah default klasik ISC/NEIC dari model Jeffreys-Bullen. Daftarnya ditentukan
# dari histogram di bawah, bukan dari asumsi.
DEFAULT_DEPTHS = (10.0, 35.0)

depth_counts = df["depth"].value_counts()
fixed_depth_share = df["depth"].isin(DEFAULT_DEPTHS).mean()
print(f"Event di kedalaman default {DEFAULT_DEPTHS}: {fixed_depth_share:.2%}")
if fixed_depth_share > 0.10:
    print("  -> Populasi kedalaman default besar. Laporkan ini sebagai limitasi:")
    print("     kanal kedalaman dangkal sebagian adalah artefak solusi yang hilang.")
    print("     Bagian 7.4 mengukur seberapa besar pengaruhnya ke hasil.")

print("\nNilai kedalaman yang paling sering muncul:")
print(depth_counts.head(12).to_string())

# Setiap kandidat dibandingkan dengan tetangga terdekatnya, supaya pemilihan nilai
# default bisa diaudit dan bukan sekadar menebak mana yang terlihat besar.
print("\nUji lonjakan: hitungan dibanding rata-rata tetangga +-2 km")
for cand in (10.0, 33.0, 35.0):
    nb = [depth_counts.get(cand + d, 0) for d in (-2, -1, 1, 2)]
    base = sum(nb) / len(nb)
    n = depth_counts.get(cand, 0)
    verdict = "lonjakan" if base and n > 1.8 * base else "dalam rentang normal"
    print(f"  {cand:5.1f} km : {n:7,}  tetangga {base:8,.0f}  -> {verdict}"
          f"{'  [dipakai]' if cand in DEFAULT_DEPTHS else ''}")

In [ ]:
# Homogenitas skala magnitudo. MLE Gutenberg-Richter mengasumsikan satu skala saja.
mag_type_cols = [c for c in df_raw.columns if "mag" in c.lower() and c.lower() != "magnitude"]
print(f"Kemungkinan kolom tipe magnitudo: {mag_type_cols or 'tidak ditemukan'}")
for c in mag_type_cols:
    vc = df_raw[c].value_counts(dropna=False)
    if 1 < len(vc) <= 20:
        print(f"\n{c}:")
        print(vc.to_string())

print("\nPeriksa plot frequency-magnitude di Bagian 4, cari tangga di sekitar M 5,5,")
print("tempat konversi Ms/Mw biasanya memunculkan diskontinuitas.")

In [ ]:
print(f"Dataset instrumental bersih: {len(df):,} record "
      f"({df['year'].min():.0f}-{df['year'].max():.0f})")
print()
print(df[["latitude", "longitude", "depth", "magnitude"]].describe().round(3).to_string())

# Indikator yang dipakai pipeline bahaya.
#
# Kedalaman default diperlakukan sebagai "tidak diketahui", bukan sebagai pengukuran.
# Semuanya berada di bawah ambang 70 km, jadi tanpa perlakuan ini setiap satu dari
# mereka terhitung dangkal dan menggelembungkan kanal kedalaman dengan data hilang.
# Bagian 7.4 mengukur pengaruhnya ke hasil akhir.
_is_default = df["depth"].isin(DEFAULT_DEPTHS)

df["is_shallow"] = (df["depth"] <= 70.0).astype(int)
df["is_default_depth"] = _is_default.astype(int)
df["depth_known"] = (~_is_default).astype(int)
df["is_shallow_known"] = ((df["depth"] <= 70.0) & ~_is_default).astype(int)

print(f"Porsi event berkedalaman default            : {df['is_default_depth'].mean():.2%}")
print(f"Proporsi dangkal nasional, semua event      : {df['is_shallow'].mean():.4f}")
print(f"Proporsi dangkal, hanya kedalaman diketahui : "
      f"{df.loc[df['depth_known'] == 1, 'is_shallow'].mean():.4f}")

---
## 3. Binning Spasial H3

Memakai list comprehension di atas array NumPy, yang menghindari overhead per baris dari
`DataFrame.apply(axis=1)`. Ini bukan vektorisasi dalam arti SIMD: `h3.latlng_to_cell`
adalah pemanggilan C skalar, sekali per record. Pengukuran waktu di bawah untuk
checklist item 3.

Cell ini melakukan binning di Resolusi 5 untuk laporan cakupan. Fungsi pipeline di
Bagian 5 melakukan binning sekali lagi, karena fungsi itu harus menerima resolusi
sebagai parameter untuk analisis sensitivitas. Satu pemanggilan terduplikasi, dibiarkan
supaya tahap ini tetap terlihat.

In [ ]:
H3_RES = 5

t0 = time.perf_counter()
lats = df["latitude"].to_numpy()
lngs = df["longitude"].to_numpy()
df["h3_index"] = [h3.latlng_to_cell(la, lo, H3_RES) for la, lo in zip(lats, lngs)]
t_bin = time.perf_counter() - t0

n_cells = df["h3_index"].nunique()
print(f"Penugasan H3: {t_bin:.2f} detik untuk {len(df):,} record")
print(f"Sel heksagonal aktif di res {H3_RES}: {n_cells:,}")
print(f"Rata-rata event per sel aktif: {len(df) / n_cells:.1f}")
print(f"Median event per sel aktif: {df.groupby('h3_index').size().median():.0f}")

cell_area = h3.average_hexagon_area(H3_RES, unit="km^2")
print(f"\nLuas sel di res {H3_RES}: {cell_area:,.0f} km^2")
print(f"Total luas tercakup sel aktif: {n_cells * cell_area:,.0f} km^2")

In [ ]:
# Distribusi jumlah event per sel. Ekor panjang di sisi kiri inilah yang menjadi alasan
# keberadaan ambang inklusi di Bagian 5.
per_cell = df.groupby("h3_index").size()
print("Event per sel, kuantil:")
print(per_cell.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.99]).round(1).to_string())
for t in (5, 10, 20, 50):
    print(f"  sel dengan < {t:3d} event: {(per_cell < t).sum():6,} "
          f"({(per_cell < t).mean():5.1%})")

---
## 4. Magnitudo Kelengkapan $M_c$

$M_c$ yang di-hardcode tanpa plot pendukung adalah asumsi paling terbuka untuk diserang
dalam studi nilai *b* mana pun, jadi bagian ini menghasilkan bukti, bukan klaim.

$M_c$ diestimasi dengan metode **MAXC**: bin magnitudo yang memuat frekuensi
non-kumulatif tertinggi menandai titik balik distribusi frequency-magnitude, ditambah
koreksi $+0{,}2$ untuk bias negatif estimator yang sudah dikenal. Estimasinya diplot
berdampingan dengan nilai operasional, lalu nilai operasional itu diargumentasikan
terhadapnya.

In [ ]:
DELTA_M = 0.1   # presisi binning magnitudo katalog

mag_binned = (df["magnitude"] / DELTA_M).round() * DELTA_M
mc_maxc = mag_binned.value_counts().idxmax() + 0.2
print(f"Estimasi MAXC untuk Mc: {mc_maxc:.2f}")

MC = 4.0   # nilai operasional; diargumentasikan terhadap mc_maxc dan plot di bawah
print(f"Mc operasional:         {MC:.2f}")
if abs(MC - mc_maxc) > 0.3:
    print(f"\n  CATATAN: Mc operasional berbeda {abs(MC - mc_maxc):.2f} dari MAXC.")
    print( "  Argumentasikan pilihan ini di laporan, atau pakai nilai MAXC.")

In [ ]:
incremental = mag_binned.value_counts().sort_index()
cumulative = incremental[::-1].cumsum()[::-1]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(incremental.index, incremental.values, "o", ms=3.5,
            alpha=0.75, label="Inkremental $N(M)$")
ax.semilogy(cumulative.index, cumulative.values, "s", ms=3.5,
            alpha=0.75, label=r"Kumulatif $N(\geq M)$")
ax.axvline(MC, color="crimson", ls="--", lw=1.6, label=f"$M_c$ operasional = {MC:.1f}")
ax.axvline(mc_maxc, color="darkorange", ls=":", lw=1.6, label=f"MAXC = {mc_maxc:.2f}")
ax.axvline(5.5, color="grey", ls="-.", lw=1.0, alpha=0.6,
           label="M 5,5 (tangga konversi Ms/Mw)")
ax.set(xlabel="Magnitudo", ylabel="Jumlah event",
       title="Distribusi Frequency-Magnitude: bukti untuk $M_c$")
ax.legend(frameon=True)
plt.tight_layout()
plt.savefig(out("fig_frequency_magnitude_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Cara membaca kurva kumulatif: di atas Mc kurvanya mestinya mendekati garis lurus")
print("pada sumbu log ini. Kelengkungan di bawah Mc adalah rollover deteksi. Tangga yang")
print("terlihat di sekitar M 5,5 menandakan skala magnitudo yang tercampur.")

In [ ]:
df_mc_preview = df[df["magnitude"] >= MC]
print(f"Event di atas Mc={MC}: {len(df_mc_preview):,} dari {len(df):,} "
      f"({len(df_mc_preview) / len(df):.1%})")
print(f"Rata-rata magnitudo di atas Mc: {df_mc_preview['magnitude'].mean():.4f}")

# Nilai b baseline nasional, cadangan untuk sel yang tidak sanggup mendukung estimasi
national_b_preview = np.log10(np.e) / (df_mc_preview["magnitude"].mean() - (MC - DELTA_M / 2.0))
print(f"Nilai b baseline nasional: {national_b_preview:.4f}")
if national_b_preview < 0.6:
    print("\n  PERINGATAN: national_b dekat ke lantai clipping 0,5. Sel hasil imputasi")
    print("  akan menempel di ujung atas kanal inverted-stress. Periksa katalognya.")

---
## 5. Pipeline SHI

### 5.1 Formulasi

**Nilai *b* Aki-Utsu dengan pemulusan 1-ring.** Tegangan kerak bersifat kontinu melintasi
batas sel, sehingga sel yang jarang data mengalami kelaparan ukuran sampel dan
menghasilkan tebing batas yang artifisial. Karena itu estimasinya dihitung di atas disk
topologis $D_1(h_i) = \text{gridDisk}(h_i, 1)$, yaitu sel itu sendiri plus enam
tetangganya:

$$b = \frac{\log_{10}(e)}{\bar{M} - \left(M_c - \frac{\Delta M}{2}\right)},
\qquad \sigma_b = \frac{b}{\sqrt{N_{\text{disk}}}}$$

Pemulusan hanya berlaku untuk kanal ini. Jumlah event, rasio dangkal, dan magnitudo
maksimum adalah statistik deskriptif sel itu sendiri, bukan estimasi sebuah medan yang
kontinu secara spasial. Sel dengan $N_{\text{disk}} < 30$ mewarisi $b_{\text{national}}$
dan diberi flag `b_estimated = False`; sel dengan $\sigma_b \le 0{,}1$ diberi flag
`b_reliable`.

**Inklusi sel, $N_i \ge 10$.** Di bawah ambang itu setiap kanal menjadi tidak stabil ke
arah yang sama. Satu event dangkal tunggal memberi $S_i = 1{,}0$, nilai $b$ yang tak
terukur diimputasi di sekitar $0{,}8$ yang posisinya ada di seperlima bawah domain
clipping sehingga skor $B_{\text{inv}}$-nya tinggi, dan hasilnya sel yang hampir kosong
tersaji sebagai bahaya tinggi. Pada PRD v2.0 sel semacam itu mendapat
$\text{SHI} \approx 0{,}53$ dan masuk tier **High**. Ambang ini adalah keputusan
penilaian, dan Bagian 7 mengujinya pada $N_{\min} \in \{5, 10, 20\}$.

**Empat kanal, terbatas di $[0,1]$.** Domain rujukan ditetapkan tetap di setiap tempat
yang punya batas fisis, sehingga nilainya tidak bergeser ketika katalog bertambah atau
resolusi berubah.

| Kanal | Definisi | Rujukan |
| :-- | :-- | :-- |
| $N_{\text{norm}}$ | $\log_{10}(N_i^{M_c} + 1)$, diskalakan min-max | spesifik per run |
| $S$ | $\dfrac{N_{\text{shallow},i} + \kappa p_{\text{nat}}}{N_i + \kappa}$, $\kappa = 10$ | absolut |
| $B_{\text{inv}}$ | $1 - \dfrac{b_i - 0{,}5}{2{,}0 - 0{,}5}$ | tetap $[0{,}5\,;\,2{,}0]$ |
| $M_{\text{norm}}$ | $\text{clip}\!\left(\dfrac{M_{\max,i} - M_c}{9{,}5 - M_c}, 0, 1\right)$ | tetap $[M_c\,;\,9{,}5]$ |

Hitungan jumlah event memakai event di atas $M_c$, menyamakan basisnya dengan kanal
nilai *b*. Menghitung seluruh event akan sama saja dengan mengukur kerapatan jaringan
seismograf, bukan seismisitas, karena cakupan stasiun di Jawa dan Sumatra jauh lebih
padat daripada di Indonesia timur.

$N_{\text{norm}}$ adalah satu-satunya kanal yang bergantung pada run, jadi $L_{\min}$ dan
$L_{\max}$ dikembalikan sebagai metadata run agar perbandingan antar-resolusi tetap bisa
dilakukan.

**Komposit.** $\text{SHI}_i = \sum_k w_k x_{ik}$, tanpa post-normalisasi. PRD v2.0
menerapkan min-max terakhir yang memaksa tepat satu sel bernilai 0,0 dan satu sel
bernilai 1,0, sehingga jumlah per tier menjadi artefak penskalaan dan antar-run tidak
lagi sebanding. Karena setiap kanal sudah berada di $[0,1]$ dan $\sum w_k = 1$,
kompositnya terbatas tanpa penskalaan ulang.

Satu konsekuensi yang harus dinyatakan terang-terangan di bagian Hasil: indeks tanpa
penskalaan ulang bisa meninggalkan tier **Very High** dalam keadaan kosong. Itu temuan
tentang tingkat bahaya absolut, bukan cacat.

In [ ]:
MIN_CELL_EVENTS = 10     # ambang inklusi sel
KAPPA           = 10.0   # pseudo-count shrinkage untuk rasio dangkal
MIN_DISK_EVENTS = 30     # minimum event di atas Mc dalam disk 1-ring untuk estimasi b
B_MIN, B_MAX    = 0.5, 2.0    # domain b tektonik yang realistis secara fisis
M_CEILING       = 9.5    # magnitudo terbesar yang pernah tercatat secara instrumental

CHANNELS = ["n_norm", "s_norm", "b_inv", "m_norm"]

# right=False memberi interval tertutup di kiri [lo, hi), sesuai tabel tier yang
# dipublikasikan. Default pd.cut adalah right=True, yang akan menaruh SHI tepat 0,25 di
# Low dan 0,50 di Moderate, satu tier di bawah apa yang tabel itu nyatakan. Tepi atas
# tak hingga menjaga SHI = 1,0 tetap di dalam Very High alih-alih jatuh ke luar bin
# sebagai NaN.
TIER_EDGES  = [0.0, 0.25, 0.50, 0.75, np.inf]
MAP_BINS    = [0.0, 0.25, 0.50, 0.75, 1.0]      # folium butuh tepi yang berhingga
TIER_LABELS = ["Low", "Moderate", "High", "Very High"]

In [ ]:
def min_max(series):
    """Skalakan ke [0,1]. Series konstan dipetakan ke nol semua, bukan dibagi nol."""
    lo, hi = series.min(), series.max()
    return (series - lo) / (hi - lo) if hi > lo else series * 0.0


def entropy_weights(X):
    """Bobot entropi Shannon atas matriks (n_sel, n_kanal) bernilai [0,1].

    Dispersi rendah -> entropi tinggi -> bobot rendah. Kanal yang hampir konstan di
    seluruh sel membawa sedikit informasi dan karenanya diturunkan bobotnya, yang
    justru tidak bisa dilakukan oleh baseline bobot sama rata.
    """
    sums = X.sum(axis=0)
    P = np.divide(X, sums, out=np.zeros_like(X), where=sums > 0)
    logP = np.zeros_like(P)              # sudah dinolkan, sehingga konvensi 0*ln0 = 0
    np.log(P, out=logP, where=P > 0)     # tidak perlu penanganan khusus
    E = -(P * logP).sum(axis=0) / np.log(len(X))
    d = 1.0 - E
    d[sums <= 0] = 0.0                   # kanal degenerat berarti dispersinya nol
    assert d.sum() > 0, "semua kanal degenerat; bobot entropi tidak bisa diturunkan"
    return d / d.sum()

In [ ]:
def build_shi(data, h3_res, mc, min_cell_events=MIN_CELL_EVENTS, kappa=KAPPA,
              delta_m=DELTA_M, min_disk_events=MIN_DISK_EVENTS,
              b_min=B_MIN, b_max=B_MAX, m_ceiling=M_CEILING,
              exclude_default_depth=False):
    """Binning, agregasi, estimasi b, normalisasi dan komposit, untuk satu konfigurasi.

    Dibuat sebagai fungsi, bukan kode lurus, karena Bagian 7 menjalankan ulang seluruh
    pipeline tujuh kali untuk analisis sensitivitas. Satu sumber kebenaran lebih baik
    daripada tujuh salinan yang saling menyimpang.

    exclude_default_depth=True membuat rasio dangkal dihitung hanya dari event yang
    kedalamannya diketahui. Event berkedalaman default tetap ikut di hitungan densitas
    dan magnitudo maksimum, karena gempanya memang terjadi; yang tidak diketahui cuma
    kedalamannya. Memisahkan begini membuat yang teruji hanya kanal kedalaman, bukan
    tercampur efek menyusutnya jumlah event.

    Mengembalikan (DataFrame ringkasan, dict metadata).
    """
    # --- binning -------------------------------------------------------------
    la = data["latitude"].to_numpy()
    lo = data["longitude"].to_numpy()
    d = data.assign(h3_index=[h3.latlng_to_cell(a, o, h3_res) for a, o in zip(la, lo)])
    d_mc = d[d["magnitude"] >= mc]

    # --- agregasi per sel ----------------------------------------------------
    s = d.groupby("h3_index", as_index=False).agg(
        total_events=("magnitude", "count"),
        shallow_events=("is_shallow", "sum"),
        shallow_known=("is_shallow_known", "sum"),
        depth_known=("depth_known", "sum"),
        default_depth_events=("is_default_depth", "sum"),
        max_magnitude=("magnitude", "max"),
        mean_magnitude=("magnitude", "mean"),
    )
    # Hitungan di atas Mc, supaya kanal densitas memakai basis event yang sama
    # dengan kanal nilai b
    n_mc = d_mc.groupby("h3_index", as_index=False).agg(n_mc=("magnitude", "count"))
    s = s.merge(n_mc, on="h3_index", how="left")
    s["n_mc"] = s["n_mc"].fillna(0).astype(int)

    cells_total = len(s)
    s = s[s["total_events"] >= min_cell_events].reset_index(drop=True)
    assert len(s) > 0, f"tidak ada sel yang lolos min_cell_events={min_cell_events}"

    # Porsi kedalaman default per sel: diagnostik, bukan kanal. Dibawa ke tabel Top-20
    # supaya pembaca langsung tahu seberapa jauh kanal kedalaman sel itu bisa dipercaya.
    s["default_depth_share"] = s["default_depth_events"] / s["total_events"]

    # --- rasio dangkal tersusut: menarik sel ber-N kecil ke arah laju nasional
    if exclude_default_depth:
        num, den = s["shallow_known"], s["depth_known"]
        p_nat = d.loc[d["depth_known"] == 1, "is_shallow"].mean()
    else:
        num, den = s["shallow_events"], s["total_events"]
        p_nat = d["is_shallow"].mean()

    # Shrinkage menangani sel yang seluruh event-nya berkedalaman default: den = 0,
    # sehingga rasionya jatuh ke laju nasional alih-alih membagi nol.
    s["shallow_ratio"] = (num + kappa * p_nat) / (den + kappa)

    # --- nilai b Aki-Utsu dengan pemulusan 1-ring ----------------------------
    national_b = np.log10(np.e) / (d_mc["magnitude"].mean() - (mc - delta_m / 2.0))

    # Lookup dibangun dari seluruh katalog di atas Mc, bukan dari daftar sel yang sudah
    # disaring, sehingga tetangga yang terbuang oleh ambang inklusi masih menyumbang
    # ke penjumlahan disk.
    stats = d_mc.groupby("h3_index").agg(n=("magnitude", "count"),
                                         sm=("magnitude", "sum"))
    n_lk = stats["n"].to_dict()
    s_lk = stats["sm"].to_dict()

    # Beberapa ribu sel x 7 lookup dict. Loop ini murah; biaya sebenarnya ada di
    # ekspor GeoJSON dan rendering folium di bawah.
    b_vals, b_errs = [], []
    for cell in s["h3_index"]:
        disk = h3.grid_disk(cell, 1)
        n_disk = sum(n_lk.get(x, 0) for x in disk)
        sum_mag = sum(s_lk.get(x, 0.0) for x in disk)
        if n_disk >= min_disk_events:
            b = np.log10(np.e) / (sum_mag / n_disk - (mc - delta_m / 2.0))
            b = float(np.clip(b, b_min, b_max))
            b_vals.append(b)
            b_errs.append(b / np.sqrt(n_disk))
        else:
            b_vals.append(np.nan)
            b_errs.append(np.nan)

    s["b_value"] = b_vals
    s["b_std_error"] = b_errs
    s["b_estimated"] = s["b_value"].notna()
    s["b_value_imputed"] = s["b_value"].fillna(national_b)
    s["b_reliable"] = s["b_std_error"] <= 0.1   # NaN <= 0.1 bernilai False, sesuai maksud

    # --- empat kanal ---------------------------------------------------------
    s["log_events"] = np.log10(s["n_mc"] + 1.0)
    s["n_norm"] = min_max(s["log_events"])                  # bergantung pada run
    s["s_norm"] = s["shallow_ratio"]                        # sudah [0,1]
    s["b_inv"] = 1.0 - (s["b_value_imputed"] - b_min) / (b_max - b_min)
    s["m_norm"] = ((s["max_magnitude"] - mc) / (m_ceiling - mc)).clip(0.0, 1.0)

    # --- komposit, tanpa post-normalisasi ------------------------------------
    X = s[CHANNELS].to_numpy(dtype=float)
    w_equal = np.full(len(CHANNELS), 1.0 / len(CHANNELS))
    w_entropy = entropy_weights(X)

    s["shi"] = X @ w_equal
    s["shi_entropy"] = X @ w_entropy
    s["shi_rank_pct"] = s["shi"].rank(pct=True)
    s["hazard_level"] = pd.cut(s["shi"], bins=TIER_EDGES, labels=TIER_LABELS,
                               right=False).astype(str)

    # Tier kedua, berbasis kuartil, untuk keperluan prioritisasi.
    #
    # Tier absolut di atas memakai ambang tetap 0,25/0,50/0,75 dan itu benar sebagai
    # pernyataan tingkat bahaya. Tapi karena seluruh Indonesia bertektonik aktif,
    # sebagian besar sel menumpuk di satu tier dan jadi tidak berguna untuk memilih
    # mana yang ditangani lebih dulu. Kuartil menjawab pertanyaan yang berbeda:
    # "relatif terhadap sel lain di Indonesia, sel ini ada di seperempat mana?"
    #
    # Diturunkan dari shi_rank_pct, bukan pd.qcut, karena peringkat persentil sudah
    # seragam menurut definisinya sehingga tidak ada masalah ambang duplikat.
    s["hazard_quartile"] = pd.cut(
        s["shi_rank_pct"], bins=[0.0, 0.25, 0.50, 0.75, 1.0],
        labels=["Q1 (terendah)", "Q2", "Q3", "Q4 (tertinggi)"],
        include_lowest=True).astype(str)

    meta = {
        "h3_res": h3_res, "mc": mc, "min_cell_events": min_cell_events,
        "kappa": kappa, "cells_total": cells_total, "cells_kept": len(s),
        "national_b": float(national_b), "p_shallow_national": float(p_nat),
        "L_min": float(s["log_events"].min()), "L_max": float(s["log_events"].max()),
        "w_equal": w_equal, "w_entropy": w_entropy,
        "b_min": b_min, "b_max": b_max, "m_ceiling": m_ceiling,
        "exclude_default_depth": exclude_default_depth,
    }
    return s, meta

### 5.2 Run baseline di Resolusi 5

In [ ]:
t0 = time.perf_counter()
h3_summary, META = build_shi(df, h3_res=H3_RES, mc=MC)
print(f"Pipeline: {time.perf_counter() - t0:.1f} detik\n")

print(f"Sel dipertahankan:       {META['cells_kept']:,} dari {META['cells_total']:,} "
      f"(terbuang {META['cells_total'] - META['cells_kept']:,} "
      f"dengan < {MIN_CELL_EVENTS} event)")
print(f"Nilai b baseline nasional: {META['national_b']:.4f}")
print(f"Proporsi dangkal nasional: {META['p_shallow_national']:.4f}")
print(f"b terukur:               {h3_summary['b_estimated'].sum():,} sel "
      f"({h3_summary['b_estimated'].mean():.1%})")
print(f"b andal (sigma <= 0,1):  {h3_summary['b_reliable'].sum():,} sel")

# Metadata run untuk perbandingan kanal densitas antar-resolusi
print(f"\nPenskalaan kanal densitas: L_min={META['L_min']:.4f}, L_max={META['L_max']:.4f}")

In [ ]:
print("Bobot kanal")
print(f"{'kanal':>10} {'sama rata':>11} {'entropi':>9} {'std dev':>9}")
for name, we, wh in zip(CHANNELS, META["w_equal"], META["w_entropy"]):
    print(f"{name:>10} {we:11.4f} {wh:9.4f} {h3_summary[name].std():9.4f}")

print("\nKorelasi antar-kanal. n_norm dan m_norm terkopel secara konstruksi:")
print("magnitudo maksimum adalah statistik nilai ekstrem, jadi ia naik seiring ukuran sampel.")
print(h3_summary[CHANNELS].corr().round(3).to_string())

# Pembobotan entropi menghargai dispersi, dan itu bukan hal yang sama dengan menghargai
# relevansi fisis. Nilai b Indonesia berkerumun di pita yang sempit, sehingga b_inv bisa
# terdorong ke bobot nyaris nol dan diam-diam mengeluarkan kanal tegangan tektonik dari
# indeks. Ditandai di sini, daripada membiarkan tabel sensitivitas berkesan bahwa kedua
# skema pembobotan sepakat.
starved = [n for n, w in zip(CHANNELS, META["w_entropy"]) if w < 0.02]
if starved:
    print(f"\nPERINGATAN: pembobotan entropi nyaris menghapus {starved}.")
    print("Varians rendah bukan berarti tidak relevan. Perlakukan run entropi sebagai")
    print("uji sensitivitas dan pertahankan bobot sama rata sebagai baseline yang")
    print("dilaporkan, atau argumentasikan pembuangan kanal itu secara seismologis.")

In [ ]:
print(f"Rentang SHI (bobot sama rata): [{h3_summary['shi'].min():.4f}, "
      f"{h3_summary['shi'].max():.4f}]")
print(f"Rentang SHI (bobot entropi):   [{h3_summary['shi_entropy'].min():.4f}, "
      f"{h3_summary['shi_entropy'].max():.4f}]")
print(f"Rata-rata SHI: {h3_summary['shi'].mean():.4f}   "
      f"Median: {h3_summary['shi'].median():.4f}")

tiers = (h3_summary["hazard_level"].value_counts()
         .reindex(TIER_LABELS, fill_value=0).to_frame("sel"))
tiers["persen"] = (tiers["sel"] / len(h3_summary) * 100).round(2)
print("\nDistribusi tier bahaya:")
print(tiers.to_string())

empty = [t for t in TIER_LABELS if tiers.loc[t, "sel"] == 0]
if empty:
    print(f"\nTier kosong: {empty}. Ini temuan tentang tingkat bahaya absolut di bawah")
    print("bobot sama rata, bukan cacat. Laporkan apa adanya, jangan diskalakan ulang.")

---
## 6. Validasi

Dijalankan sebelum ekspor apa pun. Notebook ini seharusnya gagal di sini, bukan menulis
indeks yang diam-diam rusak ke disk.

Assertion terakhir secara khusus menjaga mode kegagalan PRD v2.0. Sel tanpa event di
atas $M_c$ punya $M_{\max} < M_c$, sehingga $M_{\text{norm}}$ terpotong menjadi 0 dan
$N_{\text{norm}}$ bernilai 0, menyisakan $\text{SHI} = 0{,}25(S + B_{\text{inv}})$. Untuk
$b_{\text{national}}$ yang masuk akal, batas itu berada di sekitar 0,45, di bawah ambang
**High**. Kalau assertion ini menyala, $b_{\text{national}}$ sudah runtuh ke arah lantai
clipping dan katalognya perlu diperiksa, dan itu layak dihentikan.

In [ ]:
def validate(s, meta, verbose=True):
    """Pemeriksaan invariant. Melempar error pada apa pun yang merusak indeks terekspor."""
    assert s["shi"].between(0.0, 1.0).all(), "SHI di luar [0,1]"
    assert s["shi"].notna().all(), "ada NaN di SHI"
    assert s["shi_entropy"].between(0.0, 1.0).all(), "SHI bobot entropi di luar [0,1]"
    assert s["total_events"].ge(meta["min_cell_events"]).all(), \
        "ada sel di bawah ambang inklusi yang lolos"
    assert s["b_value_imputed"].between(meta["b_min"], meta["b_max"]).all(), \
        "b di luar domain clipping"
    assert s[CHANNELS].apply(lambda c: c.between(0.0, 1.0).all()).all(), \
        "ada sub-indeks yang keluar dari [0,1]"
    assert s["shallow_ratio"].between(0.0, 1.0).all(), "rasio dangkal tersusut di luar batas"
    assert np.isclose(meta["w_entropy"].sum(), 1.0), "bobot entropi tidak berjumlah 1"
    assert s["h3_index"].is_unique, "ada h3_index duplikat"
    assert s["hazard_level"].ne("nan").all(), "ada nilai SHI yang tidak terbin"

    # Batas tier tertutup di kiri, sesuai yang dipublikasikan
    assert s.loc[s["shi"] < 0.25, "hazard_level"].eq("Low").all()
    assert s.loc[s["shi"].between(0.25, 0.50, inclusive="left"),
                 "hazard_level"].eq("Moderate").all()
    assert s.loc[s["shi"].between(0.50, 0.75, inclusive="left"),
                 "hazard_level"].eq("High").all()
    assert s.loc[s["shi"] >= 0.75, "hazard_level"].eq("Very High").all()

    # Mode kegagalan v2.0: sel yang hampir kosong menumpang b hasil imputasi dan rasio
    # dangkal tanpa shrinkage untuk masuk ke tier High.
    empty_mc = s[s["n_mc"] == 0]
    assert empty_mc["shi"].lt(0.50).all(), (
        f"{empty_mc['shi'].ge(0.50).sum()} sel tanpa event di atas Mc mencapai High; "
        f"national_b={meta['national_b']:.3f} mungkin sudah runtuh ke arah lantai"
    )

    if verbose:
        print(f"Semua invariant terpenuhi pada {len(s):,} sel.")
    return True


validate(h3_summary, META)

In [ ]:
# Bukan error, tapi baca daftar ini sebelum memercayai Top-20: sel jarang data yang
# nilai b-nya hasil imputasi namun tetap skor High karena satu event besar.
watch = h3_summary[(h3_summary["total_events"] < 20)
                   & (~h3_summary["b_estimated"])
                   & (h3_summary["shi"] >= 0.50)]
if len(watch):
    print(f"CATATAN: {len(watch)} sel jarang data dengan b imputasi skor High. Periksa:")
    print(watch[["h3_index", "total_events", "n_mc", "max_magnitude", "shi"]]
          .to_string(index=False))
else:
    print("Tidak ada sel jarang data dengan b imputasi yang mencapai tier High.")

In [ ]:
# Centroid, untuk plot statis dan laporan tabular
centroids = [h3.cell_to_latlng(c) for c in h3_summary["h3_index"]]
h3_summary["centroid_lat"] = [c[0] for c in centroids]
h3_summary["centroid_lon"] = [c[1] for c in centroids]

# Pemicu risiko dominan: kanal mana yang membuat sel ini menonjol.
#
# Dihitung dari skor terstandardisasi, bukan dari nilai kanal mentah. Mengambil idxmax
# atas nilai mentah hanya akan memilih kanal yang rata-rata nasionalnya paling tinggi,
# bukan kanal yang khas untuk sel itu: dengan mean b_inv 0,85 dan mean m_norm 0,21,
# b_inv akan hampir selalu menang dan m_norm tidak akan pernah menang sekali pun,
# terlepas dari isi selnya. Yang informatif adalah simpangan dari rata-rata.
_z = (h3_summary[CHANNELS] - h3_summary[CHANNELS].mean()) / h3_summary[CHANNELS].std()
h3_summary["risk_driver"] = _z.idxmax(axis=1).map({
    "n_norm": "densitas event",
    "s_norm": "kedalaman dangkal",
    "b_inv":  "nilai b rendah (tegangan)",
    "m_norm": "magnitudo maksimum",
})
print(h3_summary["risk_driver"].value_counts().to_string())

---
## 7. Analisis Sensitivitas

Empat pemeriksaan, masing-masing menjawab pertanyaan yang pasti diajukan penguji.

1. **Ambang inklusi.** $N_{\min} \in \{5, 10, 20\}$ adalah keputusan penilaian, jadi
   seberapa banyak hasilnya bertumpu pada ambang itu?
2. **Skema pembobotan.** Bobot sama rata versus bobot entropi Shannon, diukur dengan
   korelasi peringkat dan tumpang-tindih Top-20.
3. **Resolusi.** $h \in \{4, 5, 6\}$. Karena tiga dari empat kanal sekarang dinormalisasi
   terhadap domain tetap, nilai SHI sebanding antar-run begitu $L_{\min}$ dan $L_{\max}$
   dilaporkan.
4. **Kedalaman default.** Sebagian besar katalog memberi kedalaman 10 km ketika inversi
   kedalaman gagal, dan di katalog ini porsinya besar. Menghitung semuanya sebagai
   dangkal menggelembungkan $S$ dengan data hilang, jadi run pembanding menghitung
   $S$ hanya dari event yang kedalamannya diketahui.

   Event berkedalaman default tetap masuk hitungan densitas dan magnitudo maksimum,
   karena gempanya memang terjadi. Yang tidak diketahui cuma kedalamannya. Dengan
   memisahkan begitu, yang teruji murni kanal kedalaman, bukan tercampur efek
   menyusutnya jumlah event. Pemeriksaan ini juga melaporkan `default_depth_share` tiap
   sel Top-20, sehingga pembaca tahu baris mana yang perlu dibaca dengan hati-hati.

Bagian ini menjalankan ulang pipeline penuh tujuh kali, jadi perkirakan beberapa menit.
Resolusi 6 yang paling lambat: jumlah selnya kira-kira tujuh kali Resolusi 5.

In [ ]:
def summarise(s, meta, label):
    tiers = s["hazard_level"].value_counts().reindex(TIER_LABELS, fill_value=0)
    return {
        "konfigurasi": label,
        "sel": len(s),
        "terbuang": meta["cells_total"] - meta["cells_kept"],
        "shi_min": round(s["shi"].min(), 4),
        "shi_max": round(s["shi"].max(), 4),
        "shi_mean": round(s["shi"].mean(), 4),
        "b_terukur_pct": round(s["b_estimated"].mean() * 100, 1),
        **{t: int(tiers[t]) for t in TIER_LABELS},
    }


top20_base = set(h3_summary.nlargest(20, "shi")["h3_index"])

In [ ]:
# 7.1 Ambang inklusi
# Nama variabelnya spesifik, bukan `rows`: 7.3 di bawah juga membangun tabel, dan nama
# yang sama akan menimpa hasil ini sehingga tabel N_min hilang dari sesi.
rows_nmin = []
for nmin in (5, 10, 20):
    s_n, m_n = build_shi(df, h3_res=H3_RES, mc=MC, min_cell_events=nmin)
    validate(s_n, m_n, verbose=False)
    r = summarise(s_n, m_n, f"N_min = {nmin}")
    r["overlap_top20"] = len(top20_base & set(s_n.nlargest(20, "shi")["h3_index"]))
    rows_nmin.append(r)

nmin_table = pd.DataFrame(rows_nmin)
print("Pengaruh ambang inklusi sel")
print(nmin_table.to_string(index=False))
nmin_table.to_csv(out("sensitivity_nmin.csv"), index=False)
print("\nTumpang-tindih Top-20 dihitung terhadap run baseline N_min = 10. Tumpang-tindih")
print("yang tinggi berarti peringkat utamanya tidak bergantung pada ambang itu.")

In [ ]:
# 7.2 Skema pembobotan
rho = h3_summary["shi"].corr(h3_summary["shi_entropy"], method="spearman")
top20_entropy = set(h3_summary.nlargest(20, "shi_entropy")["h3_index"])

print(f"Spearman rho (sama rata vs entropi): {rho:.4f}")
print(f"Tumpang-tindih Top-20:               {len(top20_base & top20_entropy)}/20")
print()
print("Pergeseran bobot:")
for name, we, wh in zip(CHANNELS, META["w_equal"], META["w_entropy"]):
    print(f"  {name:>10}: {we:.4f} -> {wh:.4f}  ({(wh - we) / we * 100:+6.1f}%)")

tier_cross = pd.crosstab(
    h3_summary["hazard_level"],
    pd.cut(h3_summary["shi_entropy"], bins=TIER_EDGES, labels=TIER_LABELS,
           right=False).astype(str),
    rownames=["sama rata"], colnames=["entropi"],
)
print("\nPerpindahan tier antar-skema pembobotan:")
print(tier_cross.to_string())

In [ ]:
# 7.3 Resolusi H3
rows_res = []
for res in (4, 5, 6):
    s_r, m_r = build_shi(df, h3_res=res, mc=MC)
    validate(s_r, m_r, verbose=False)
    r = summarise(s_r, m_r, f"res {res}")
    r["sel_km2"] = round(h3.average_hexagon_area(res, unit="km^2"), 1)
    r["L_min"] = round(m_r["L_min"], 3)
    r["L_max"] = round(m_r["L_max"], 3)
    rows_res.append(r)

res_table = pd.DataFrame(rows_res)
print("Pengaruh resolusi H3")
print(res_table.to_string(index=False))
print("\nNilai SHI sebanding antar-baris di atas untuk tiga dari empat kanal. Kanal")
print("densitas diskalakan ulang per run, jadi bandingkan lewat L_min / L_max.")
res_table.to_csv(out("sensitivity_resolution.csv"), index=False)

In [ ]:
# 7.4 Kedalaman default
s_dd, m_dd = build_shi(df, h3_res=H3_RES, mc=MC, exclude_default_depth=True)
validate(s_dd, m_dd, verbose=False)

top20_dd = set(s_dd.nlargest(20, "shi")["h3_index"])
overlap = len(top20_base & top20_dd)

dd_table = pd.DataFrame([
    summarise(h3_summary, META, "semua kedalaman"),
    summarise(s_dd, m_dd, "kedalaman diketahui"),
])
print("Pengaruh memperlakukan kedalaman 10 km sebagai tidak diketahui")
print(dd_table.to_string(index=False))
dd_table.to_csv(out("sensitivity_default_depth.csv"), index=False)

print(f"\nRata-rata s_norm : {h3_summary['s_norm'].mean():.4f} -> {s_dd['s_norm'].mean():.4f}")
print(f"Rata-rata SHI    : {h3_summary['shi'].mean():.4f} -> {s_dd['shi'].mean():.4f}")
print(f"Tumpang-tindih Top-20 : {overlap}/20")

rho_dd = (h3_summary.set_index("h3_index")["shi"]
          .reindex(s_dd["h3_index"])
          .corr(s_dd.set_index("h3_index")["shi"].reindex(s_dd["h3_index"]),
                method="spearman"))
print(f"Spearman rho antar-kedua run : {rho_dd:.4f}")

In [ ]:
# Seberapa banyak Top-20 baseline yang bertumpu pada kedalaman default
dd_check = (h3_summary.nlargest(20, "shi")
            [["h3_index", "total_events", "default_depth_share", "shallow_ratio",
              "shi", "risk_driver"]]
            .copy())
dd_check["bertahan_top20"] = dd_check["h3_index"].isin(top20_dd)
dd_shi = s_dd.set_index("h3_index")["shi"]
dd_check["shi_kedalaman_diketahui"] = dd_check["h3_index"].map(dd_shi).round(4)
dd_check["delta"] = (dd_check["shi_kedalaman_diketahui"] - dd_check["shi"]).round(4)

print("Top-20 baseline, diperiksa terhadap run kedalaman-diketahui")
print(dd_check.round(4).to_string(index=False))

print(f"\nPorsi kedalaman default di Top-20: "
      f"min {dd_check['default_depth_share'].min():.1%}, "
      f"median {dd_check['default_depth_share'].median():.1%}, "
      f"maks {dd_check['default_depth_share'].max():.1%}")
print(f"Bertahan di Top-20 baru: {dd_check['bertahan_top20'].sum()}/20")

heavy = dd_check[dd_check["default_depth_share"] > 0.5]
if len(heavy):
    print(f"\n{len(heavy)} sel Top-20 lebih dari separuh event-nya berkedalaman default.")
    print("Kanal kedalaman untuk sel-sel itu sebagian besar mengukur data hilang, dan")
    print("peringkatnya harus diberi catatan di laporan, bukan disajikan apa adanya.")

---
## 8. Visualisasi Statis

Tiga panel pada extent yang sama: densitas event logaritmik, medan nilai *b* yang sudah
dimuluskan, dan indeks komposit. Diplot sebagai centroid sel, bukan poligon, karena
lebih cepat dan memadai pada skala ini. Nilai *b* terukur dan hasil imputasi digambar
terpisah, supaya panel tengah tidak menyajikan imputasi sebagai pengukuran.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5.4), sharex=True, sharey=True)
pt = dict(s=7, marker="o", linewidths=0)

# Panel 1: densitas event logaritmik
p0 = axes[0].scatter(h3_summary["centroid_lon"], h3_summary["centroid_lat"],
                     c=h3_summary["log_events"], cmap="viridis", **pt)
axes[0].set_title(r"Densitas event log, $\log_{10}(N^{M_c} + 1)$")
fig.colorbar(p0, ax=axes[0], shrink=0.82)

# Panel 2: nilai b, hanya sel yang terukur
meas = h3_summary[h3_summary["b_estimated"]]
imp = h3_summary[~h3_summary["b_estimated"]]
axes[1].scatter(imp["centroid_lon"], imp["centroid_lat"],
                c="lightgrey", label=f"imputasi (n={len(imp):,})", **pt)
p1 = axes[1].scatter(meas["centroid_lon"], meas["centroid_lat"],
                     c=meas["b_value"], cmap="coolwarm_r", vmin=0.5, vmax=1.5, **pt)
axes[1].set_title(r"Nilai $b$ termuluskan (1-ring). Biru = $b$ rendah = tegangan tinggi")
axes[1].legend(loc="lower left", fontsize=8, markerscale=2)
fig.colorbar(p1, ax=axes[1], shrink=0.82)

# Panel 3: indeks komposit, pada skala tier yang dipublikasikan
p2 = axes[2].scatter(h3_summary["centroid_lon"], h3_summary["centroid_lat"],
                     c=h3_summary["shi"], cmap="YlOrRd", vmin=0.0, vmax=1.0, **pt)
axes[2].set_title("Seismic Hazard Index (bobot sama rata)")
fig.colorbar(p2, ax=axes[2], shrink=0.82, ticks=MAP_BINS)

for ax in axes:
    ax.set(xlabel="Bujur", ylabel="Lintang")
    ax.set_aspect("equal", adjustable="box")

plt.tight_layout()
plt.savefig(out("fig_hazard_panels.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))

axes[0].hist(h3_summary["shi"], bins=50, color="#c0392b", alpha=0.85)
for edge in MAP_BINS[1:-1]:
    axes[0].axvline(edge, color="black", ls="--", lw=1.0)
axes[0].set(xlabel="SHI", ylabel="Jumlah sel", title="Distribusi SHI dengan ambang tier")

axes[1].hist(meas["b_value"], bins=40, color="#2980b9", alpha=0.85)
axes[1].axvline(META["national_b"], color="crimson", ls="--",
                label=f"b nasional = {META['national_b']:.3f}")
axes[1].set(xlabel=r"Nilai $b$", ylabel="Jumlah sel",
            title=r"Nilai $b$ terukur (terpotong ke $[0{,}5\,;\,2{,}0]$)")
axes[1].legend(fontsize=8)

sns.boxplot(data=h3_summary, x="risk_driver", y="shi", ax=axes[2])
axes[2].set(xlabel="", ylabel="SHI", title="SHI menurut pemicu risiko dominan")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig(out("fig_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Ekspor GeoJSON dan Spatial Join Administratif

Karena filter spasialnya berupa kotak pembatas, sebagian besar sel berbahaya tinggi
berada di lepas pantai di atas Palung Sunda dan tidak beririsan dengan provinsi mana
pun. Sel-sel itu diberi label provinsi terdekat dan ditandai `offshore`, bukan dibiarkan
NaN atau diam-diam disajikan seolah berada di darat.

Heksagon Resolusi 5 menutupi sekitar 253 km² dan masih bisa melintasi batas provinsi,
sehingga `n_provinces` mencatat berapa provinsi yang disentuhnya. Untuk sel-sel itu
labelnya menyebut satu provinsi dari beberapa.

In [ ]:
def cell_to_polygon(cell):
    boundary = h3.cell_to_boundary(cell)
    return Polygon([[p[1], p[0]] for p in boundary])   # GeoJSON mau (lon, lat)


h3_summary["geometry"] = h3_summary["h3_index"].apply(cell_to_polygon)
gdf_h3 = gpd.GeoDataFrame(h3_summary, geometry="geometry", crs="EPSG:4326")

# hazard_level sudah str, bukan Categorical: driver GeoJSON menolak dtype kategorikal
# pada beberapa versi pyogrio.
assert gdf_h3["hazard_level"].dtype == object

OUT_GEOJSON = out("indonesia_seismic_hazard_h3_res5.geojson")
t0 = time.perf_counter()
gdf_h3.to_file(OUT_GEOJSON, driver="GeoJSON")
print(f"Menulis {len(gdf_h3):,} sel ke {OUT_GEOJSON} "
      f"dalam {time.perf_counter() - t0:.1f} detik "
      f"({os.path.getsize(OUT_GEOJSON) / 1e6:.1f} MB)")

# Pemeriksaan baca-ulang, checklist item 5
_back = gpd.read_file(OUT_GEOJSON)
assert len(_back) == len(gdf_h3), "baca-ulang GeoJSON kehilangan baris"
assert {"b_estimated", "b_reliable", "shi", "hazard_level"} <= set(_back.columns), \
    "baca-ulang GeoJSON kehilangan kolom flag"
print("Pemeriksaan baca-ulang lolos.")

In [ ]:
top20 = gdf_h3.nlargest(20, "shi").copy()

if ADM_PATH:
    gdf_adm = gpd.read_file(ADM_PATH)
    assert "NAME_1" in gdf_adm.columns, (
        f"{ADM_PATH} bukan GADM level 1: kolom NAME_1 tidak ada. "
        f"Kolom yang tersedia: {list(gdf_adm.columns)}"
    )
    gdf_adm = gdf_adm[["NAME_1", "geometry"]].rename(columns={"NAME_1": "province"})
    print(f"Batas administratif: {len(gdf_adm)} provinsi, CRS {gdf_adm.crs}")

    onshore = gpd.sjoin(top20, gdf_adm, how="inner", predicate="intersects")
    onshore["n_provinces"] = onshore.groupby("h3_index")["province"].transform("size")
    onshore["location_type"] = "onshore"
    onshore = onshore.drop(columns="index_right").drop_duplicates(subset="h3_index")

    offshore = top20[~top20["h3_index"].isin(onshore["h3_index"])].copy()
    if len(offshore):
        # Join terdekat butuh CRS terproyeksi; EPSG:3857 memadai untuk memeringkat
        # berdasarkan kedekatan pada skala ini.
        nearest = (gpd.sjoin_nearest(offshore.to_crs(3857), gdf_adm.to_crs(3857),
                                     how="left")
                   .to_crs(4326)
                   .drop(columns="index_right")
                   .drop_duplicates(subset="h3_index"))
        nearest["n_provinces"] = 0
        nearest["location_type"] = "offshore (provinsi terdekat)"
        top20_adm = pd.concat([onshore, nearest])
    else:
        top20_adm = onshore

    top20_adm = top20_adm.sort_values("shi", ascending=False)
    print(f"Komposisi Top-20: {top20_adm['location_type'].value_counts().to_dict()}")
    assert top20_adm["province"].notna().all(), "ada sel Top-20 tanpa label provinsi"
    assert len(top20_adm) == 20, f"diharapkan 20 baris, dapat {len(top20_adm)}"
else:
    print("Batas administratif tidak ada: spatial join dilewati, pipeline tetap lanjut.")
    print("Lihat sel resolusi path di Bagian 1 untuk cara melengkapinya.")
    top20_adm = top20.assign(province=pd.NA, location_type="belum di-join", n_provinces=0)

In [ ]:
REPORT_COLS = ["h3_index", "province", "location_type", "n_provinces",
               "centroid_lat", "centroid_lon", "total_events", "n_mc",
               "shallow_ratio", "default_depth_share", "max_magnitude",
               "b_value_imputed", "b_estimated", "b_reliable", "shi",
               "shi_rank_pct", "hazard_level", "hazard_quartile", "risk_driver"]

top20_out = pd.DataFrame(top20_adm[REPORT_COLS])
top20_out.to_csv(out("top_20_critical_hexagons.csv"), index=False)
print("Menulis top_20_critical_hexagons.csv\n")

print(top20_out[["h3_index", "province", "location_type", "total_events",
                 "max_magnitude", "b_value_imputed", "b_estimated",
                 "shi", "hazard_level", "risk_driver"]]
      .round(4).to_string(index=False))

---
## 10. Choropleth Interaktif

Ambang kelasnya adalah ambang tier, sehingga warna peta dan label tabel tidak mungkin
bertentangan. Default folium adalah binning kuantil, yang justru akan bertentangan
dengan tier yang dipublikasikan.

Catatan biaya: cell ini memuat GeoJSON yang sama dua kali, sekali untuk isian dan sekali
untuk lapisan hover, dan ini cell paling lambat di notebook. Kalau sampai tidak
tertangani, batasi lapisan tooltip ke sel berbahaya tinggi saja, atau buang lapisan itu
dan pertahankan choropleth-nya.

In [ ]:
m = folium.Map(location=[-2.5, 118.0], zoom_start=5, tiles=None)

# Riwayat singkat pemilihan peta dasar, supaya tidak diulang lagi.
#
# CartoDB dark_matter kini berada di balik API key. Permintaan tanpa kunci tetap dijawab
# HTTP 200, tetapi isinya ubin bertuliskan "API KEY REQUIRED", jadi gagalnya senyap.
# Terbukti dari isi piksel: ubin Carto cuma memuat 16 warna unik.
#
# OpenStreetMap sempat dipakai sebagai cadangan dan ternyata lebih buruk: servernya
# menjawab 403 "Access blocked" untuk berkas HTML lokal, karena kebijakan penggunaan
# ubin mereka menolak aplikasi yang tidak teridentifikasi. Ubin blokirnya opak putih,
# jadi ia justru menutupi lapisan di bawahnya.
#
# Keduanya dibuang. Esri Canvas bebas kunci dan sudah diuji di dua belas ubin yang
# menutupi Indonesia: semuanya 200 dengan isi nyata, 140 sampai 372 warna unik. Dark
# Gray luminans rata-ratanya 43,5 berbanding 40,1 milik Carto, jadi nuansanya setara.
ESRI = ("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/"
        "{name}/MapServer/tile/{{z}}/{{y}}/{{x}}")   # ArcGIS memakai z/y/x, bukan z/x/y

# show=False itu wajib, bukan hiasan. Tanpanya folium menggambar SEMUA TileLayer
# sekaligus, dan yang ditambahkan terakhir menutupi yang pertama, berapa pun isi
# kontrol lapisannya.
folium.TileLayer(tiles=ESRI.format(name="World_Dark_Gray_Base"),
                 attr="Tiles &copy; Esri", name="Esri Dark Gray",
                 overlay=False, control=True).add_to(m)
folium.TileLayer(tiles=ESRI.format(name="World_Light_Gray_Base"),
                 attr="Tiles &copy; Esri", name="Esri Light Gray",
                 overlay=False, control=True, show=False).add_to(m)

folium.Choropleth(
    geo_data=OUT_GEOJSON,
    name="Seismic Hazard Index",
    data=h3_summary,
    columns=["h3_index", "shi"],
    key_on="feature.properties.h3_index",
    fill_color="YlOrRd",
    bins=MAP_BINS,            # ambang yang sama dengan tabel tier bahaya
    fill_opacity=0.75,
    line_opacity=0.25,
    legend_name="Seismic Hazard Index (SHI)",
).add_to(m)

style_function = lambda x: {
    "fillColor": "#ffffff", "color": "#000000", "fillOpacity": 0.05, "weight": 0.2,
}
highlight_function = lambda x: {
    "fillColor": "#00ffff", "color": "#ffffff", "fillOpacity": 0.4, "weight": 1.5,
}

tooltip = folium.features.GeoJson(
    OUT_GEOJSON,
    style_function=style_function,
    control=False,
    highlight_function=highlight_function,
    tooltip=folium.features.GeoJsonTooltip(
        fields=["h3_index", "n_mc", "max_magnitude", "b_value_imputed",
                "b_estimated", "shi", "hazard_level", "risk_driver"],
        aliases=["Indeks H3:", "Event (>=Mc):", "Mag maks:", "Nilai b (1-ring):",
                 "b terukur:", "SHI:", "Kelas risiko:", "Pemicu:"],
        style=("background-color: #1e1e1e; color: #ffffff; font-family: monospace; "
               "font-size: 11px; padding: 8px; border-radius: 4px;"),
    ),
)
m.add_child(tooltip)

# Penanda Top-20, supaya peringkat tabularnya bisa dicari di peta
for _, r in top20_out.iterrows():
    folium.CircleMarker(
        location=[r["centroid_lat"], r["centroid_lon"]],
        radius=4, color="#00ffff", fill=True, fill_opacity=0.9, weight=1,
        popup=folium.Popup(
            f"<b>{r['h3_index']}</b><br>SHI {r['shi']:.3f} ({r['hazard_level']})"
            f"<br>{r['province']} — {r['location_type']}"
            f"<br>pemicu: {r['risk_driver']}", max_width=280),
    ).add_to(m)

folium.LayerControl().add_to(m)

OUT_MAP = out("indonesia_seismic_hazard_map.html")
t0 = time.perf_counter()
m.save(OUT_MAP)
map_mb = os.path.getsize(OUT_MAP) / 1e6
print(f"Menyimpan {OUT_MAP} dalam {time.perf_counter() - t0:.1f} detik ({map_mb:.1f} MB)")

# Petanya sengaja TIDAK disematkan inline.
#
# display(m) menulis seluruh HTML peta ke dalam .ipynb sebagai output cell. Pada run
# nyata itu membengkakkan notebook dari 0,6 MB menjadi 22,9 MB, dan notebook sebesar itu
# lambat dibuka, berat disinkronkan, serta tidak ter-render di GitHub. Padahal yang
# bernilai dari peta ini adalah interaktivitasnya, dan itu cuma bekerja di browser.
#
# Pola spasialnya sudah tersaji statis di panel ketiga Bagian 8, yang itulah yang masuk
# ke laporan. Jadi peta ini dibiarkan sebagai file untuk dibuka sendiri.
print(f"\nBuka {OUT_MAP} di browser untuk versi interaktifnya.")
print("Tidak disematkan di notebook: ukurannya akan menggelembungkan file deliverable.")
print("Untuk laporan, pakai panel SHI di Bagian 8 yang sudah tersimpan sebagai PNG.")

---
## 11. Ringkasan Deliverable

Jalankan cell di bawah paling akhir. Isinya menyatakan ulang konfigurasi run dan
angka-angka yang dibutuhkan laporan maupun paper, termasuk hal-hal yang harus dikutip
dan bukan diasumsikan diam-diam: $M_c$ operasional beserta estimasi MAXC-nya, konstanta
penskalaan kanal densitas, dan rentang SHI yang teramati.

In [ ]:
print("=" * 70)
print("RINGKASAN RUN H3-SHI")
print("=" * 70)
print(f"Katalog                   {CSV_PATH}")
print(f"Record, mentah -> dipakai {len(df_raw):,} -> {len(df):,}")
print(f"Periode                   {df['year'].min():.0f}-{df['year'].max():.0f}")
print(f"Resolusi H3               {META['h3_res']} "
      f"({h3.average_hexagon_area(META['h3_res'], unit='km^2'):,.0f} km^2 per sel)")
print(f"Mc operasional            {META['mc']:.2f}   (estimasi MAXC {mc_maxc:.2f})")
print(f"Ambang inklusi            N >= {META['min_cell_events']}")
print(f"Sel dipertahankan         {META['cells_kept']:,} dari {META['cells_total']:,}")
print(f"Nilai b baseline nasional {META['national_b']:.4f}")
print(f"b terukur                 {h3_summary['b_estimated'].mean():.1%} dari sel")
print(f"b andal                   {h3_summary['b_reliable'].mean():.1%} dari sel")
print(f"Penskalaan densitas       L_min={META['L_min']:.4f}, L_max={META['L_max']:.4f}")
print(f"Rentang SHI               [{h3_summary['shi'].min():.4f}, "
      f"{h3_summary['shi'].max():.4f}]")
print(f"Bobot entropi             "
      + ", ".join(f"{n}={w:.3f}" for n, w in zip(CHANNELS, META["w_entropy"])))
print("-" * 70)
print("Distribusi tier")
for t in TIER_LABELS:
    n = int((h3_summary["hazard_level"] == t).sum())
    print(f"  {t:<10} {n:6,}  ({n / len(h3_summary):6.2%})")
print("-" * 70)
print("File yang ditulis")
for f in (OUT_GEOJSON, OUT_MAP, out("top_20_critical_hexagons.csv"),
          out("sensitivity_nmin.csv"), out("sensitivity_resolution.csv"),
          out("sensitivity_default_depth.csv"),
          out("fig_frequency_magnitude_distribution.png"),
          out("fig_hazard_panels.png"), out("fig_distributions.png")):
    mark = f"{os.path.getsize(f) / 1e6:.2f} MB" if os.path.exists(f) else "TIDAK ADA"
    print(f"  {f:<52} {mark}")
print("=" * 70)

---
## 12. Uji Kewajaran terhadap Gempa Besar yang Diketahui

Menampilkan Top-20 saja tidak membuktikan indeksnya masuk akal. Uji yang lebih tajam
adalah kebalikannya: ambil gempa besar yang sudah diketahui semua orang, cari selnya,
dan lihat skor apa yang diberikan indeks ini. Kalau megathrust Sunda mendapat skor
sedang, itu harus bisa dijelaskan, bukan didiamkan.

Pertanyaan yang dijawab bagian ini ada tiga.

1. **Apakah katalognya benar-benar memuat mainshock-nya?** Dicek dengan mencari
   magnitudo terbesar di dalam radius sekitar 0,5° dari episentrum menurut literatur,
   lalu membandingkannya dengan $M_w$ yang diketahui. Kalau katalog mencatat Aceh 2004
   sebagai 6,5 alih-alih 9,1, maka kanal magnitudo maksimum bekerja di atas angka yang
   tersaturasi, dan itu menjelaskan banyak hal sekaligus.
2. **Berapa skor selnya, dan kanal mana yang menahannya?** Keempat kanal disajikan
   berdampingan dengan rata-rata nasional, sehingga terlihat mana yang menarik sel itu
   ke bawah.
3. **Apakah hasilnya sensitif terhadap presisi koordinat?** Sel Resolusi 5 hanya
   sekitar 17 km, sementara koordinat episentrum dari literatur bisa bergeser sejauh
   itu. Maka yang dilaporkan bukan cuma sel yang memuat titiknya, tetapi juga SHI
   tertinggi di dalam disk 1-ring di sekitarnya. Kalau keduanya berbeda jauh,
   kesimpulannya memang bergantung pada presisi koordinat dan harus dinyatakan begitu.

Koordinat di bawah diambil dari katalog publik dan sifatnya perkiraan. Yang menjadi
bukti adalah angka yang dicetak notebook, bukan koordinatnya.

### Satu batas struktural yang bisa dihitung sebelum dijalankan

Dengan bobot sama rata, satu gempa raksasa saja tidak cukup untuk membawa sebuah sel ke
puncak peringkat, dan ini bisa ditunjukkan dengan aritmetika biasa.

Ambang bawah Top-20 pada run terakhir adalah SHI 0,7807. Sebuah sel dengan $M_w$ 9,1
mendapat $M_{\text{norm}} = 0{,}927$, yang karena bobotnya 0,25 menyumbang 0,232. Sisanya
harus datang dari tiga kanal lain, yaitu 0,549, atau rata-rata 0,732 per kanal. Rata-rata
nasional ketiga kanal itu hanya 0,676.

Jadi sel yang memuat gempa terbesar dalam sejarah instrumental Indonesia, tetapi
biasa-biasa saja pada kerapatan kejadian, kedalaman, dan nilai *b*, **tetap tidak akan
masuk Top-20**. Peringkat teratas ditempati sel yang di atas rata-rata pada beberapa
kanal sekaligus, dan itulah yang terjadi pada sel-sel Maluku Utara.

Apakah sifat ini benar tergantung pertanyaan yang diajukan. Untuk "di mana bahaya seismik
kumulatif paling tinggi", sifat ini masuk akal. Untuk "di mana gempa paling merusak bisa
terjadi", sifat ini salah, dan $M_{\text{norm}}$ perlu bobot lebih besar atau indeks yang
berbeda. Nyatakan pilihan ini di bagian Diskusi, karena inilah alasan struktural kenapa
megathrust Sunda bisa absen dari Top-20, terlepas dari ada atau tidaknya saturasi
magnitudo.

In [ ]:
# Apakah event terbesar memang ada di katalog ini, dan tercatat sebagai apa
MAG_TYPE_COL = next((c for c in df_raw.columns
                     if c.lower() in ("magnitude type", "magnitude_type")), None)

big_cols = ["year", "latitude", "longitude", "depth", "magnitude"]
if MAG_TYPE_COL:
    big_cols.append(MAG_TYPE_COL)

print(f"Magnitudo terbesar di katalog tersaring: {df['magnitude'].max():.2f}")
print(f"Nilai yang dipakai sebagai plafon m_norm: {M_CEILING:.2f}\n")
print("Lima belas event terbesar:")
print(df.nlargest(15, "magnitude")[big_cols].to_string(index=False))

In [ ]:
# nama, lintang, bujur, Mw menurut literatur
REFERENCE_EVENTS = [
    ("Sumatra-Andaman 2004-12-26",  3.316,  95.854, 9.1),
    ("Nias 2005-03-28",             2.085,  97.108, 8.6),
    ("Padang 2009-09-30",          -0.725,  99.856, 7.6),
    ("Yogyakarta 2006-05-26",      -7.961, 110.446, 6.3),
    ("Lombok 2018-08-05",          -8.258, 116.439, 6.9),
    ("Palu 2018-09-28",            -0.178, 119.840, 7.5),
]

idx = h3_summary.set_index("h3_index")
top20_floor = h3_summary.nlargest(20, "shi")["shi"].min()

rows_ref = []
for name, lat, lon, mw_lit in REFERENCE_EVENTS:
    cell = h3.latlng_to_cell(lat, lon, H3_RES)

    # Magnitudo terbesar yang dicatat katalog di sekitar titik itu, dalam +-0,5 derajat
    near = df[(df["latitude"] - lat).abs().le(0.5)
              & (df["longitude"] - lon).abs().le(0.5)]
    mag_cat = near["magnitude"].max() if len(near) else np.nan
    if len(near) and MAG_TYPE_COL:
        mtype = df_raw.loc[near["magnitude"].idxmax(), MAG_TYPE_COL]
    else:
        mtype = None

    r = {"event": name, "mw_literatur": mw_lit,
         "mag_katalog": round(mag_cat, 2) if len(near) else None,
         "tipe_mag": mtype, "event_sekitar": len(near), "sel_ada": cell in idx.index}

    if r["sel_ada"]:
        c = idx.loc[cell]
        r.update({
            "total_events": int(c["total_events"]), "n_mc": int(c["n_mc"]),
            "max_mag_sel": round(c["max_magnitude"], 2),
            "b_value": round(c["b_value_imputed"], 4),
            "b_terukur": bool(c["b_estimated"]),
            "n_norm": round(c["n_norm"], 4), "s_norm": round(c["s_norm"], 4),
            "b_inv": round(c["b_inv"], 4), "m_norm": round(c["m_norm"], 4),
            "shi": round(c["shi"], 4), "persentil": round(c["shi_rank_pct"], 4),
            "tier": c["hazard_level"], "pemicu": c["risk_driver"],
        })
        disk = [idx.loc[x, "shi"] for x in h3.grid_disk(cell, 1) if x in idx.index]
        r["shi_maks_1ring"] = round(max(disk), 4)
    rows_ref.append(r)

ref_table = pd.DataFrame(rows_ref)
ref_table.to_csv(out("validation_reference_events.csv"), index=False)
print(f"Ambang bawah Top-20 (SHI): {top20_floor:.4f}\n")
print(ref_table.to_string(index=False))

In [ ]:
# Diagnosis: kanal mana yang menahan tiap event, dan apakah magnitudonya tersaturasi
nat = h3_summary[CHANNELS].mean()
print("Rata-rata nasional: " + "  ".join(f"{c}={nat[c]:.4f}" for c in CHANNELS))
print(f"Ambang bawah Top-20: SHI {top20_floor:.4f}\n")

for r in rows_ref:
    print(f"{r['event']}")
    if not r["sel_ada"]:
        print(f"  Selnya tidak masuk indeks: < {MIN_CELL_EVENTS} event atau di luar bbox.\n")
        continue

    gap = top20_floor - r["shi"]
    print(f"  SHI {r['shi']:.4f} ({r['tier']}, persentil {r['persentil']:.1%}), "
          f"maks 1-ring {r['shi_maks_1ring']:.4f}, selisih ke Top-20 {gap:+.4f}")

    lemah = [c for c in CHANNELS if r[c] < nat[c]]
    print("  Kanal di bawah rata-rata nasional: "
          + (", ".join(f"{c} {r[c]:.3f} vs {nat[c]:.3f}" for c in lemah) if lemah
             else "tidak ada"))

    # Saturasi magnitudo: mainshock yang dikenal versus yang dicatat katalog
    if r["mag_katalog"] is not None:
        shortfall = r["mw_literatur"] - r["mag_katalog"]
        if shortfall > 0.5:
            print(f"  MAGNITUDO TERSATURASI: literatur Mw {r['mw_literatur']}, "
                  f"katalog {r['mag_katalog']} (tipe {r['tipe_mag']}), "
                  f"kurang {shortfall:.1f}")
            print(f"    m_norm sel ini {r['m_norm']:.3f}. Dengan Mw sebenarnya nilainya "
                  f"{min(max((r['mw_literatur'] - MC) / (M_CEILING - MC), 0), 1):.3f}, "
                  f"jadi SHI-nya kehilangan sekitar "
                  f"{0.25 * (min(max((r['mw_literatur'] - MC) / (M_CEILING - MC), 0), 1) - r['m_norm']):.4f}")
    print()

In [ ]:
# Apakah koreksi saturasi saja cukup mengangkat event-event ini ke Top-20
print("Hipotesis: jika m_norm dihitung dari Mw literatur, bukan dari magnitudo katalog")
print(f"{'event':<28} {'shi':>8} {'shi_koreksi':>12} {'cukup Top-20?':>15}")
for r in rows_ref:
    if not r["sel_ada"] or r["mag_katalog"] is None:
        continue
    m_fix = min(max((r["mw_literatur"] - MC) / (M_CEILING - MC), 0.0), 1.0)
    shi_fix = r["shi"] + 0.25 * (m_fix - r["m_norm"])
    print(f"{r['event']:<28} {r['shi']:8.4f} {shi_fix:12.4f} "
          f"{'ya' if shi_fix >= top20_floor else 'belum':>15}")

print("\nKalau kolom shi_koreksi tetap di bawah ambang Top-20, maka saturasi magnitudo")
print("bukan satu-satunya penyebab, dan kanal lain yang menahannya. Lihat daftar kanal")
print("di bawah rata-rata nasional pada sel sebelumnya.")

In [ ]:
# 12.5 Apakah kotak pembatas memotong rupture Aceh 2004?
#
# Sel episentrum Aceh cuma berisi belasan gempa, padahal itu gempa terbesar dalam
# sejarah instrumental Indonesia. Dugaannya: rupture 2004 memanjang jauh ke utara
# sepanjang Busur Andaman, sementara filter spasial memotong di 6 LU.
#
# Diuji langsung dari df_raw, sebelum penyaringan, supaya yang terbuang ikut terhitung.
LINTANG_POTONG = 6.0
KORIDOR_BUJUR = (92.0, 100.0)      # koridor Sumatra-Andaman

kor = df_raw[df_raw["longitude"].between(*KORIDOR_BUJUR)].copy()
kor["thn"] = pd.to_numeric(kor["Year"], errors="coerce")
kor = kor[(kor["thn"] >= 1960) & kor["latitude"].notna()]

print(f"Koridor bujur {KORIDOR_BUJUR[0]}-{KORIDOR_BUJUR[1]} BT, tahun >= 1960: "
      f"{len(kor):,} event\n")

pita = pd.cut(kor["latitude"],
              bins=[-11, 0, 3, 6, 9, 12, 15, 20],
              labels=["-11..0", "0..3", "3..6", "6..9", "9..12", "12..15", "15..20"])
tabel = (kor.groupby(pita, observed=False)
         .agg(total=("latitude", "size"), mag_maks=("magnitude", "max")))
tabel["di_dalam_bbox"] = [i < 3 for i in range(len(tabel))]
print("Sebaran event menurut pita lintang:")
print(tabel.to_string())

atas = kor[kor["latitude"] > LINTANG_POTONG]
print(f"\nDi atas {LINTANG_POTONG} LU, jadi TERBUANG : {len(atas):,} event "
      f"({len(atas)/len(kor):.1%} dari koridor)")

In [ ]:
# Sekuens gempa susulan 2004-2005 secara khusus: ini yang paling menentukan
seq = kor[kor["thn"].between(2004, 2005)]
atas_seq = seq[seq["latitude"] > LINTANG_POTONG]

print(f"Event koridor pada 2004-2005 : {len(seq):,}")
print(f"  di dalam bbox (<= {LINTANG_POTONG} LU) : {len(seq) - len(atas_seq):,}")
print(f"  terbuang    (>  {LINTANG_POTONG} LU) : {len(atas_seq):,} "
      f"({len(atas_seq)/len(seq):.1%})" if len(seq) else "  (tidak ada event)")

if len(seq) and len(atas_seq) / len(seq) > 0.25:
    print("\nTERKONFIRMASI: sebagian besar sekuens 2004-2005 ada di luar wilayah studi.")
    print("Sel Aceh terlihat sepi karena kotak pembatasnya, bukan karena katalognya.")
    print(f"Untuk mencakupnya, batas utara perlu dinaikkan ke sekitar "
          f"{np.ceil(seq['latitude'].max()):.0f} LU.")
    print("Itu keputusan metodologis, bukan sekadar mengganti angka: wilayah studi")
    print("melebar ke Laut Andaman yang tektoniknya berbeda dari busur Sunda.")
elif len(seq):
    print("\nTIDAK terkonfirmasi: sebagian besar sekuensnya ada di dalam bbox.")
    print("Sel Aceh yang sepi berarti soal kelengkapan katalog, bukan batas wilayah.")

---
## 13. Homogenitas Skala Magnitudo

Bagian 12 menunjukkan katalog ini mencatat Aceh 2004 sebagai 7,1 padahal $M_w$-nya 9,1,
dan selisihnya mengikuti tipe magnitudo. Bagian ini menindaklanjutinya: seberapa jauh
pencampuran skala merusak hasil, dan apakah bisa diperbaiki.

MLE Aki-Utsu mengasumsikan satu skala magnitudo. Katalog ini memuat dua belas, dan yang
benar-benar $M_w$ hanya 1.344 dari 314 ribu.

### Kenapa tidak langsung dikonversi saja

Tiga jalan yang jelas, dan dua di antaranya buntu.

**Membatasi ke subset $M_w$ saja.** Hanya 1.344 event di seluruh Indonesia. Ambang
$N_{\text{disk}} \ge 30$ tidak akan terpenuhi di hampir semua sel. Tetap dijalankan di
bawah sebagai uji batas, bukan sebagai jalur utama.

**Quantile mapping ke distribusi $M_w$.** Terlihat menarik karena murni berbasis data,
tetapi keliru di sini. Lembaga seismologi menghitung $M_w$ secara selektif untuk gempa
besar, sehingga subset $M_w$ bias ke atas. Memetakan kuantil MLv ke kuantil $M_w$ akan
menggelembungkan seluruh magnitudo kecil. Asumsi bahwa kedua populasi mencuplik
distribusi yang sama tidak terpenuhi, jadi teknik ini dibuang.

**Relasi konversi.** Jalan yang lazim, tetapi koefisiennya datang dari literatur dan
tidak bisa diverifikasi dari dalam katalog ini. Dipakai sebagai cadangan, dengan
koefisien yang ditulis eksplisit agar bisa diganti, dan diberi tanda bahwa harus
diperiksa sebelum dipublikasikan.

### Yang dikerjakan di sini

Diagnostiknya dulu, karena bagian itu murni dari data dan tidak bergantung asumsi apa
pun. Baru setelah itu koreksinya.

1. **Nilai $b$ per tipe magnitudo.** Kalau mb dan MLv memberi $b$ yang berbeda jauh pada
   $M_c$ yang sama, campurannya memang bermasalah.
2. **Komposisi tipe secara spasial.** Ini kekhawatiran yang sebenarnya. Kalau Maluku
   Utara didominasi satu tipe dan Sumatra tipe lain, maka peta $b$-mu sebagian adalah
   peta lembaga mana yang melaporkan apa, bukan peta tegangan tektonik.
3. **Mencari pasangan lintas-tipe.** Katalog ini gabungan dari banyak sumber. Kalau ada
   event yang sama tercatat dua kali dengan tipe berbeda, pasangan itu memberi relasi
   konversi empiris khusus katalog ini, tanpa perlu koefisien dari luar.
4. **Run homogenisasi dan perbandingan**, lengkap dengan uji mandiri: setelah dikoreksi,
   nilai $b$ antar-tipe seharusnya saling mendekat. Kalau tidak, koreksinya gagal dan
   itu harus dilaporkan apa adanya.

### Hasil yang sudah bisa dihitung di muka: konversinya tidak akan menolong

Relasi Scordilis untuk mb hanya berlaku sampai $m_b = 6{,}2$, dan arahnya selalu
menaikkan karena titik netralnya ada di 6,87, di luar rentang itu. Diterapkan pada enam
event rujukan Bagian 12, hasilnya:

| Event | Tipe | Literatur | Katalog | Selisih awal | Sesudah konversi | Sisa |
| :-- | :-- | --: | --: | --: | --: | --: |
| Aceh 2004 | `Mw(mB)` | 9,1 | 7,1 | 2,00 | 7,10 | **2,00** |
| Nias 2005 | `Mw(mB)` | 8,6 | 7,7 | 0,90 | 7,70 | **0,90** |
| Padang 2009 | `mb` | 7,6 | 7,1 | 0,50 | 7,10 | **0,50** |
| Yogyakarta 2006 | `mb` | 6,3 | 6,0 | 0,30 | 6,13 | 0,17 |
| Lombok 2018 | `M` | 6,9 | 6,9 | 0,00 | 6,90 | 0,00 |
| Palu 2018 | `Mw` | 7,5 | 7,5 | 0,00 | 7,50 | 0,00 |

Rata-rata selisih mutlak hanya bergerak dari **0,617 ke 0,595**. Hanya Yogyakarta yang
membaik. Tiga event terbesar tidak tersentuh sama sekali: dua bertipe `Mw(mB)` yang tidak
punya relasi tepercaya, dan satu bertipe `mb` di atas plafon 6,2.

Jadi koreksi ini berhenti persis di tempat masalahnya dimulai. Bagian 13.5 harus dibaca
sebagai pengukuran seberapa sedikit yang berubah, bukan sebagai klaim bahwa katalognya
sudah terhomogenisasi.

### Jalan yang realistis

Karena konversi generik buntu, yang masuk akal untuk dikerjakan adalah **menambal
magnitudo event besar dari katalog otoritatif**. Jumlah gempa $M \ge 6{,}5$ di wilayah
dan periode ini hanya ratusan, jadi mengambil $M_w$ dari GCMT atau USGS untuk event-event
itu layak dikerjakan dan langsung menyasar sumber distorsi terbesar. Event kecil boleh
dibiarkan, karena saturasi memang tidak menyentuhnya dan merekalah yang mendominasi
estimasi $b$.

Yang tidak boleh dilakukan: menjalankan konversi generik lalu menyebut katalognya sudah
terhomogenisasi.

In [ ]:
# 13.1 Nilai b per tipe magnitudo, pada Mc yang sama agar sebanding
assert MAG_TYPE_COL, "katalog tanpa kolom tipe magnitudo, bagian ini tidak berlaku"

def b_mle(mags, mc=MC, delta_m=DELTA_M, n_min=50):
    """b Aki-Utsu untuk sekumpulan magnitudo. NaN kalau sampelnya terlalu kecil.

    Mengembalikan nilai terpotong dan nilai mentah sekaligus. Selisih di antara
    keduanya penting: kalau clipping aktif, nilai b yang dilaporkan bukan hasil
    pengukuran melainkan batas domain.
    """
    m = mags[mags >= mc]
    if len(m) < n_min:
        return np.nan, np.nan, np.nan, len(m)
    b_raw = np.log10(np.e) / (m.mean() - (mc - delta_m / 2.0))
    return float(np.clip(b_raw, B_MIN, B_MAX)), float(b_raw), b_raw / np.sqrt(len(m)), len(m)

rows_type = []
for t, grp in df.groupby(df[MAG_TYPE_COL].fillna("(tanpa tipe)")):
    b, b_raw, se, n_mc_t = b_mle(grp["magnitude"])
    mean_atas = grp.loc[grp["magnitude"] >= MC, "magnitude"].mean() if n_mc_t else np.nan
    rows_type.append({
        "tipe": t, "n_total": len(grp), "n_di_atas_Mc": n_mc_t,
        "mag_maks": round(grp["magnitude"].max(), 2),
        "mean_di_atas_Mc": round(mean_atas, 3) if n_mc_t else None,
        # Untuk GR murni, mean - Mc = 0,4343/b. Nilai jauh di atas ~0,6 menandakan
        # subsetnya TIDAK lengkap pada Mc ini, melainkan cuplikan terpilih yang besar.
        "mean_minus_Mc": round(mean_atas - MC, 3) if n_mc_t else None,
        "b": round(b, 4) if b == b else None,
        "b_tanpa_clip": round(b_raw, 4) if b_raw == b_raw else None,
        "terpotong": bool(b == b and abs(b - b_raw) > 1e-9),
        "sigma_b": round(se, 4) if se == se else None,
    })

type_table = (pd.DataFrame(rows_type)
              .sort_values("n_total", ascending=False)
              .reset_index(drop=True))
print("Nilai b per tipe magnitudo, semuanya pada Mc = {:.1f}".format(MC))
print(type_table.to_string(index=False))

# Tipe yang tidak lengkap pada Mc ini tidak boleh dibandingkan begitu saja
TIDAK_LENGKAP = 0.8
curiga = type_table[(type_table["mean_minus_Mc"].notna())
                    & (type_table["mean_minus_Mc"] > TIDAK_LENGKAP)]
if len(curiga):
    print(f"\nTipe yang kemungkinan TIDAK lengkap pada Mc = {MC}:")
    print(curiga[["tipe", "n_total", "mean_minus_Mc", "b", "b_tanpa_clip"]].to_string(index=False))
    print("Lembaga hanya menghitung magnitudo momen untuk gempa besar, jadi subset itu")
    print("adalah cuplikan terpilih, bukan katalog lengkap. Nilai b-nya tidak bermakna")
    print("di sini dan tidak boleh dipakai sebagai pembanding.")

layak = type_table[(type_table["b"].notna())
                   & (type_table["mean_minus_Mc"] <= TIDAK_LENGKAP)
                   & (type_table["n_di_atas_Mc"] >= 500)]
print(f"\nPembanding yang layak, yaitu tipe yang lengkap pada Mc = {MC} dan cukup besar:")
print(layak[["tipe", "n_di_atas_Mc", "b"]].to_string(index=False))
if len(layak) > 1:
    lo, hi = layak["b"].min(), layak["b"].max()
    print(f"\nSebaran b antar-tipe: {lo:.4f} sampai {hi:.4f}, rentang {hi - lo:.4f}")
    print(f"b gabungan seluruh katalog: {META['national_b']:.4f}")
    print("\nRentang selebar ini berarti b yang dilaporkan ikut bergantung pada tipe mana")
    print("yang kebetulan mendominasi sebuah sel, bukan hanya pada tektoniknya.")

In [ ]:
# 13.2 Apakah komposisi tipe bervariasi secara spasial, dan apakah b mengikutinya
tipe_per_sel = (df.assign(_t=df[MAG_TYPE_COL].fillna("(tanpa tipe)"))
                .groupby(["h3_index", "_t"]).size().unstack(fill_value=0))
share = tipe_per_sel.div(tipe_per_sel.sum(axis=1), axis=0)

dom = pd.DataFrame({
    "tipe_dominan": share.idxmax(axis=1),
    "porsi_dominan": share.max(axis=1),
})
for t in [c for c in ("mb", "MLv", "Mw", "M") if c in share.columns]:
    dom[f"porsi_{t}"] = share[t]

sel = h3_summary.set_index("h3_index").join(dom)
print("Tipe dominan per sel, di antara sel yang masuk indeks:")
print(sel["tipe_dominan"].value_counts().to_string())
print(f"\nPorsi tipe dominan: median {sel['porsi_dominan'].median():.2%}, "
      f"min {sel['porsi_dominan'].min():.2%}, maks {sel['porsi_dominan'].max():.2%}")

terukur = sel[sel["b_estimated"]]
print("\nNilai b menurut tipe dominan sel (hanya sel dengan b terukur):")
print(terukur.groupby("tipe_dominan")["b_value"]
      .agg(["count", "mean", "std"]).round(4).to_string())

korelasi = [c for c in sel.columns if c.startswith("porsi_") and c != "porsi_dominan"]
if korelasi:
    print("\nKorelasi porsi tipe terhadap nilai b terukur:")
    print(terukur[korelasi + ["b_value"]].corr()["b_value"].drop("b_value")
          .round(3).to_string())
    print("\nKorelasi yang kuat di sini adalah temuan serius: artinya peta b sebagian")
    print("memetakan lembaga pelapor, bukan tegangan tektonik.")

In [ ]:
# 13.3 Mencari event yang sama tercatat lebih dari sekali dengan tipe berbeda.
# Kalau ada, relasi konversinya bisa diturunkan dari katalog ini sendiri, tanpa
# meminjam koefisien dari luar.
stamp = None
for kolom in (["Date", "Time_UTC"], ["Date"], ["time"]):
    if all(c in df.columns for c in kolom):
        teks = df[kolom[0]].astype(str)
        if len(kolom) > 1:
            teks = teks + " " + df[kolom[1]].astype(str)
        coba = pd.to_datetime(teks, errors="coerce", format="mixed")
        if coba.notna().mean() > 0.5:
            stamp, dipakai = coba, kolom
            break

if stamp is None:
    print("Timestamp tidak bisa diparse dengan andal; pencarian pasangan dilewati.")
    pairs = pd.DataFrame()
else:
    print(f"Timestamp dari {dipakai}, terparse {stamp.notna().mean():.1%}")
    kunci = pd.DataFrame({
        "t": stamp.dt.floor("1min"),
        "la": df["latitude"].round(1),
        "lo": df["longitude"].round(1),
        "tipe": df[MAG_TYPE_COL].fillna("(tanpa tipe)"),
        "mag": df["magnitude"],
    }).dropna(subset=["t"])

    grup = kunci.groupby(["t", "la", "lo"])
    kandidat = grup.filter(lambda g: g["tipe"].nunique() > 1)
    print(f"Baris dalam grup dengan lebih dari satu tipe: {len(kandidat):,}")

    pasangan = []
    for _, g in kandidat.groupby(["t", "la", "lo"]):
        per_tipe = g.groupby("tipe")["mag"].mean()
        for a in per_tipe.index:
            for b_ in per_tipe.index:
                if a < b_:
                    pasangan.append({"tipe_a": a, "tipe_b": b_,
                                     "mag_a": per_tipe[a], "mag_b": per_tipe[b_]})
    pairs = pd.DataFrame(pasangan)
    if len(pairs):
        print(f"\nPasangan lintas-tipe ditemukan: {len(pairs):,}")
        print(pairs.groupby(["tipe_a", "tipe_b"]).size()
              .sort_values(ascending=False).head(10).to_string())
    else:
        print("\nTidak ada pasangan lintas-tipe: katalognya sudah dideduplikasi.")
        print("Konversi empiris tidak mungkin, jatuh ke relasi literatur di 13.4.")

In [ ]:
# 13.4 Relasi konversi ke Mw.
#
# PERIKSA DULU SEBELUM DIPUBLIKASIKAN. Koefisien di bawah berasal dari Scordilis (2006)
# dan bersifat global, bukan khusus Indonesia. Dimuat eksplisit supaya bisa diganti
# dengan relasi regional tanpa menyentuh kode lain.
#
# Tipe yang tidak punya relasi tepercaya dibiarkan apa adanya dan dihitung terpisah.
# Membiarkannya lebih jujur daripada menebak, tetapi karena MLv saja sudah seratus ribu
# event, porsi yang tidak terkoreksi harus dilaporkan terang-terangan.
#
# Mw(mB) dan Mwc TIDAK dimasukkan sebagai identitas meski namanya mengandung "Mw".
# Bagian 12 membuktikan justru tipe itulah yang paling tersaturasi: Aceh 2004 kurang
# 2,0 dan Nias 2005 kurang 0,9, keduanya Mw(mB). Memperlakukannya sebagai Mw asli akan
# menghitungnya sebagai "sudah benar" dan menutupi distorsi terbesar di katalog ini.
KONVERSI_MW = {
    # tipe: daftar (batas_bawah, batas_atas, a, b)  ->  Mw = a*M + b
    "mb": [(3.5, 6.2, 0.85, 1.03)],                               # Scordilis (2006)
    "MS": [(3.0, 6.1, 0.67, 2.07), (6.2, 8.2, 0.99, 0.08)],       # Scordilis (2006)
    "Mw": [(0.0, 10.0, 1.0, 0.0)],                                # sudah Mw
}

def ke_mw(mag, tipe):
    for lo, hi, a, b in KONVERSI_MW.get(tipe, []):
        if lo <= mag <= hi:
            return a * mag + b, True
    return mag, False

tipe_kolom = df[MAG_TYPE_COL].fillna("(tanpa tipe)")
hasil = [ke_mw(m, t) for m, t in zip(df["magnitude"].to_numpy(), tipe_kolom.to_numpy())]

df_hom = df.copy()
df_hom["magnitude"] = [h[0] for h in hasil]
df_hom["mag_dikonversi"] = [h[1] for h in hasil]

print(f"Event dikonversi : {df_hom['mag_dikonversi'].mean():.1%} "
      f"({df_hom['mag_dikonversi'].sum():,} dari {len(df_hom):,})")
print(f"Tidak terkoreksi : {(~df_hom['mag_dikonversi']).sum():,}")
print("\nTipe yang tidak terkoreksi:")
print(tipe_kolom[~df_hom["mag_dikonversi"]].value_counts().head(8).to_string())
print(f"\nMagnitudo maksimum: {df['magnitude'].max():.2f} -> {df_hom['magnitude'].max():.2f}")

# Yang paling penting diperiksa: event bertipe terdaftar yang magnitudonya jatuh DI LUAR
# rentang keberlakuan relasinya. Relasi Scordilis berhenti di mb 6,2, sementara justru
# gempa di atas itulah yang paling tersaturasi. Kalau angkanya besar, maka koreksi ini
# melewatkan persis event yang paling bermasalah.
punya_relasi = tipe_kolom.isin(KONVERSI_MW)
luar_rentang = punya_relasi & ~df_hom["mag_dikonversi"]
print(f"\nBertipe terdaftar tapi di luar rentang keberlakuan: {luar_rentang.sum():,}")
if luar_rentang.any():
    print(df.loc[luar_rentang, "magnitude"].describe()[["count", "min", "max"]].round(2).to_string())
    print(f"  di antaranya M >= 6,5 : {(df.loc[luar_rentang, 'magnitude'] >= 6.5).sum():,}")
    print("  Relasi konversinya berhenti sebelum rentang magnitudo besar, jadi event")
    print("  yang paling tersaturasi justru tidak tersentuh koreksi ini.")

In [ ]:
# Sisa saturasi setelah konversi, diukur langsung dari pasangan event rujukan Bagian 12.
# Inilah satu-satunya data berpasangan yang tersedia di sini: magnitudo katalog
# berdampingan dengan Mw dari literatur untuk event yang sama.
ref_pairs = pd.DataFrame([r for r in rows_ref if r.get("mag_katalog") is not None]).copy()
ref_pairs["selisih_awal"] = ref_pairs["mw_literatur"] - ref_pairs["mag_katalog"]
ref_pairs["mag_terkoreksi"] = [round(ke_mw(m, t)[0], 2) for m, t
                               in zip(ref_pairs["mag_katalog"], ref_pairs["tipe_mag"])]
ref_pairs["sisa_selisih"] = (ref_pairs["mw_literatur"] - ref_pairs["mag_terkoreksi"]).round(2)

print("Saturasi sebelum dan sesudah konversi, pada event rujukan")
print(ref_pairs[["event", "tipe_mag", "mw_literatur", "mag_katalog",
                 "selisih_awal", "mag_terkoreksi", "sisa_selisih"]].to_string(index=False))

awal, sisa = ref_pairs["selisih_awal"].abs().mean(), ref_pairs["sisa_selisih"].abs().mean()
print(f"\nRata-rata selisih mutlak: {awal:.3f} -> {sisa:.3f} satuan magnitudo")
print(f"Catatan: n = {len(ref_pairs)}, terlalu kecil untuk menjadi kalibrasi.")
print("Angka ini mengukur besarnya masalah, bukan menyelesaikannya. Untuk kalibrasi")
print("yang layak, perluas daftar event rujukan di Bagian 12 dengan katalog GCMT atau")
print("USGS, lalu cocokkan per tipe magnitudo.")

In [ ]:
# Uji mandiri: kalau konversinya bekerja, nilai b antar-tipe harus saling mendekat.
rows_after = []
for t, grp in df_hom.groupby(tipe_kolom.values):
    b, _, _, n = b_mle(grp["magnitude"])
    rows_after.append({"tipe": t, "n_di_atas_Mc": n, "b_sesudah": round(b, 4) if b == b else None})

banding = (type_table[["tipe", "b"]].rename(columns={"b": "b_sebelum"})
           .merge(pd.DataFrame(rows_after)[["tipe", "b_sesudah"]], on="tipe", how="left"))
print("Nilai b per tipe, sebelum dan sesudah konversi")
print(banding.to_string(index=False))

seb = banding["b_sebelum"].dropna()
ses = banding["b_sesudah"].dropna()
if len(seb) > 1 and len(ses) > 1:
    r_seb, r_ses = seb.max() - seb.min(), ses.max() - ses.min()
    print(f"\nRentang b antar-tipe: {r_seb:.4f} -> {r_ses:.4f}")
    if r_ses < r_seb:
        print("Konversi menyempitkan sebaran antar-tipe, jadi arahnya benar.")
    else:
        print("PERINGATAN: sebaran tidak menyempit. Konversinya tidak memperbaiki")
        print("homogenitas, dan hasil 13.5 harus dibaca sebagai batas atas pengaruh,")
        print("bukan sebagai perbaikan.")

In [ ]:
# 13.5 Jalankan ulang pipeline dengan magnitudo terhomogenisasi, lalu bandingkan.
#
# Mc harus diestimasi ULANG. Mc = 4,0 ditentukan pada skala magnitudo yang asli, dan
# konversi menggeser skala itu: relasi mb memetakan 4,0 menjadi 0,85*4,0 + 1,03 = 4,43.
# Memakai Mc yang sama pada dua skala berbeda berarti membandingkan dua tingkat
# kelengkapan yang berbeda, dan perbandingannya tidak sah.
#
# Arah perubahannya tidak bisa diduga di muka. Relasi Scordilis hanya berlaku di jendela
# [3,5 ; 6,2], sehingga katalog hasilnya campuran event terkonversi dan tidak, bukan GR
# yang diskalakan bersih, dan tidak ada b' teoretis yang rapi. Pada simulasi GR murni
# ambang yang tidak digeser justru menaikkan b, berlawanan dengan yang terjadi di
# katalog ini. Besarnya pun sebanding dengan efek yang sedang diukur, jadi baris
# pembanding di bawah dicetak justru untuk memperlihatkan jebakan itu.
# MAXC TIDAK bisa dipakai di sini, dan itu sendiri sebuah temuan. Hanya 20,8% event
# yang terkonversi, jadi modus distribusi gabungan masih ditentukan oleh 79% yang tidak
# tersentuh, dan MAXC mengembalikan nilai yang sama seperti sebelum konversi. Pada run
# pertama hal itu membuat koreksi ini tidak melakukan apa-apa.
#
# Yang benar: ambangnya dihitung dari relasi konversinya sendiri. Setiap tipe yang
# dikonversi memindahkan Mc = 4,0 ke tempat baru, dan agar sampel gabungannya sah,
# ambang global harus setinggi ambang tertinggi di antara mereka.
mag_binned_hom = (df_hom["magnitude"] / DELTA_M).round() * DELTA_M
mc_hom_maxc = mag_binned_hom.value_counts().idxmax() + 0.2

ambang_per_tipe = {}
for t, aturan in KONVERSI_MW.items():
    for lo, hi, a, b in aturan:
        if lo <= MC <= hi:
            ambang_per_tipe[t] = a * MC + b
            break
ambang_per_tipe["(tidak dikonversi)"] = MC

MC_HOM = float(np.ceil(max(ambang_per_tipe.values()) * 10) / 10)

print(f"Mc pada skala asli : {MC:.2f}")
print(f"MAXC pada skala terkonversi : {mc_hom_maxc:.2f}  <- menyesatkan, lihat catatan")
print("\nAmbang kelengkapan yang tersirat dari tiap relasi:")
for t, v in sorted(ambang_per_tipe.items(), key=lambda kv: -kv[1]):
    print(f"  {t:<20} {v:.3f}")
print(f"\nMc global yang dipakai : {MC_HOM:.2f}")
print("Diambil yang tertinggi, karena sampel gabungan baru sah kalau setiap subsetnya")
print("sudah lengkap pada ambang itu.")
print(f"\nHarga yang harus dibayar: menaikkan Mc dari {MC:.1f} ke {MC_HOM:.2f} membuang")
print("sebagian besar katalog. Dengan b sekitar 0,7, porsi event yang tersisa kira-kira")
print(f"10^(-0,7 x {MC_HOM - MC:.1f}) = {10 ** (-0.7 * (MC_HOM - MC)):.0%}. Perhatikan kolom")
print("b_terukur_pct di tabel bawah: penurunannya adalah inti persoalannya.")
print("\nInilah alasan sebenarnya kenapa konversi generik buntu untuk katalog ini.")
print("Pilihannya cuma dua, dan dua-duanya buruk: pakai Mc yang sah tapi buang sebagian")
print("besar data, atau pertahankan data dengan ambang yang tidak sah.\n")

s_hom, m_hom = build_shi(df_hom, h3_res=H3_RES, mc=MC_HOM)
validate(s_hom, m_hom, verbose=False)

# Run ketiga, sengaja memakai Mc yang salah, untuk menunjukkan besarnya pengaruh
# ketidakcocokan ambang itu sendiri
s_bad, m_bad = build_shi(df_hom, h3_res=H3_RES, mc=MC)

# Run keempat: katalog ASLI pada Mc yang dinaikkan. Ini yang memisahkan dua efek.
#
# Baris "terhomogenisasi (Mc 4,80)" mengubah dua hal sekaligus, skala magnitudo DAN
# ambang kelengkapan. Tanpa run ini, tidak bisa dibedakan mana yang menggeser hasil.
# Dengan membandingkan asli@4,0 -> asli@4,80 -> terhomogenisasi@4,80, pengaruh ambang
# dan pengaruh konversi bisa dibaca terpisah.
s_mc, m_mc = build_shi(df, h3_res=H3_RES, mc=MC_HOM)
validate(s_mc, m_mc, verbose=False)
print(f"b nasional dengan Mc yang tidak cocok ({MC:.1f}) : {m_bad['national_b']:.4f}")
print(f"b nasional dengan Mc yang benar ({MC_HOM:.1f})      : {m_hom['national_b']:.4f}")
print(f"b nasional pada katalog asli                : {META['national_b']:.4f}")
print("Selisih antara dua baris pertama murni artefak ambang, bukan efek homogenisasi.\n")

assert MC_HOM > MC, (
    f"MC_HOM ({MC_HOM}) tidak lebih tinggi dari MC ({MC}); koreksi ambangnya tidak "
    f"melakukan apa-apa dan dua baris pembanding di bawah akan identik")

hom_table = pd.DataFrame([
    summarise(h3_summary, META, f"asli (Mc {MC:.2f})"),
    summarise(s_mc, m_mc, f"asli (Mc {MC_HOM:.2f})"),
    summarise(s_hom, m_hom, f"terhomogenisasi (Mc {MC_HOM:.2f})"),
    summarise(s_bad, m_bad, f"terhomogenisasi (Mc {MC:.2f}, keliru)"),
])
print(hom_table.to_string(index=False))
hom_table.to_csv(out("sensitivity_magnitude_homogenisation.csv"), index=False)

top20_hom = set(s_hom.nlargest(20, "shi")["h3_index"])
bersama = h3_summary["h3_index"][h3_summary["h3_index"].isin(s_hom["h3_index"])]
rho_hom = (h3_summary.set_index("h3_index").loc[bersama, "shi"]
           .corr(s_hom.set_index("h3_index").loc[bersama, "shi"], method="spearman"))

top20_mc = set(s_mc.nlargest(20, "shi")["h3_index"])

print(f"\nb nasional   : {META['national_b']:.4f} -> {m_hom['national_b']:.4f}")
print(f"mean m_norm  : {h3_summary['m_norm'].mean():.4f} -> {s_hom['m_norm'].mean():.4f}")
print(f"mean SHI     : {h3_summary['shi'].mean():.4f} -> {s_hom['shi'].mean():.4f}")

print("\nMemisahkan pengaruh ambang dari pengaruh konversi, lewat tumpang-tindih Top-20:")
print(f"  asli@{MC:.1f} vs asli@{MC_HOM:.2f}            : "
      f"{len(top20_base & top20_mc)}/20   <- murni pengaruh ambang")
print(f"  asli@{MC_HOM:.2f} vs terhomogenisasi@{MC_HOM:.2f} : "
      f"{len(top20_mc & top20_hom)}/20   <- murni pengaruh konversi")
print(f"  asli@{MC:.1f} vs terhomogenisasi@{MC_HOM:.2f}  : "
      f"{len(top20_base & top20_hom)}/20   <- keduanya sekaligus")
print(f"\nSpearman rho (asli vs terhomogenisasi) : {rho_hom:.4f}")

if len(top20_base & top20_hom) >= 15 and rho_hom > 0.9:
    print("\nPeringkatnya bertahan. Pencampuran skala menggeser nilai absolut SHI, tetapi")
    print("tidak membalik urutan selnya, jadi kesimpulan spasialnya masih bisa dipakai")
    print("selama keterbatasan ini dinyatakan.")
else:
    print("\nPeringkatnya BERUBAH. Pencampuran skala magnitudo mempengaruhi kesimpulan")
    print("spasial, bukan hanya nilai absolutnya. Pilihan skala harus diputuskan dan")
    print("diargumentasikan sebelum hasilnya dilaporkan.")

In [ ]:
# 13.6 Uji batas: hanya magnitudo momen, tanpa konversi sama sekali
TIPE_MOMEN = ["Mw", "Mw(mB)", "Mwc"]
df_mw = df[tipe_kolom.isin(TIPE_MOMEN)]
print(f"Subset magnitudo momen: {len(df_mw):,} event "
      f"({len(df_mw)/len(df):.2%} dari katalog)")

if len(df_mw) and (df_mw["magnitude"] >= MC).sum() >= 50:
    b_mw, b_mw_raw, se_mw, n_mw = b_mle(df_mw["magnitude"])
    mean_mw = df_mw.loc[df_mw["magnitude"] >= MC, "magnitude"].mean()
    print(f"b dari subset Mw pada Mc {MC:.1f} : {b_mw:.4f} (mentah {b_mw_raw:.4f}, "
          f"n={n_mw:,})")
    print(f"mean magnitudo - Mc        : {mean_mw - MC:.3f}")
    if mean_mw - MC > 0.8:
        print("\n  Angka b di atas TIDAK bermakna. Mean yang jauh di atas Mc menunjukkan")
        print("  subset ini bukan katalog lengkap pada Mc 4,0, melainkan cuplikan event")
        print("  besar yang dipilih lembaga untuk dihitung momennya. Mc yang tepat untuk")
        print("  subset ini jauh lebih tinggi, dan dengan itu jumlah selnya makin sedikit.")

try:
    s_mw, m_mw = build_shi(df_mw, h3_res=H3_RES, mc=MC)
    print(f"\nSel yang lolos: {m_mw['cells_kept']:,} dari {m_mw['cells_total']:,}")
    print(f"b terukur: {s_mw['b_estimated'].mean():.1%}")
    print("Subset ini cukup untuk dijalankan, tapi cakupannya jauh lebih tipis.")
except AssertionError as e:
    print(f"\nTidak bisa dijalankan: {e}")
    print("Inilah jawabannya: membatasi ke Mw saja bukan pilihan yang tersedia pada")
    print("katalog ini. Pencampuran skala harus dihadapi, bukan dihindari.")

---
## 14. Berkas Review Teks

Menulis seluruh angka yang dibutuhkan laporan ke satu berkas teks kecil, dan
mencetaknya. Gunanya agar hasil run bisa dibaca, dibandingkan antar-run, atau dikirim
ke orang lain tanpa harus membuka `.ipynb` berukuran puluhan megabita.

Bagian ini sebelumnya berupa cell yang ditempel manual, dan pada satu run ia terlewat
sehingga hasilnya tidak terekam. Sekarang jadi bagian notebook, jadi `Run all` selalu
menghasilkannya.

In [ ]:
REVIEW_PATH = out("h3shi_review.txt")

lines = []
def w(s=""):
    lines.append(str(s))

w("== KONFIGURASI ==")
w(f"CSV_PATH    : {CSV_PATH}")
w(f"ADM_PATH    : {ADM_PATH}")
w(f"record      : {len(df_raw):,} -> {len(df):,}")
w(f"periode     : {df['year'].min():.0f}-{df['year'].max():.0f}")
w(f"Mc          : {MC}  (MAXC {mc_maxc:.2f})")
w(f"resolusi    : {META['h3_res']}  ({h3.average_hexagon_area(META['h3_res'], unit='km^2'):,.1f} km2)")
w(f"N_min       : {META['min_cell_events']}")
w(f"sel         : {META['cells_kept']:,} / {META['cells_total']:,}")
w(f"national_b  : {META['national_b']:.4f}")
w(f"p_shallow   : {META['p_shallow_national']:.4f}")
w(f"L_min/L_max : {META['L_min']:.4f} / {META['L_max']:.4f}")
w(f"b terukur   : {h3_summary['b_estimated'].mean():.1%}")
w(f"b andal     : {h3_summary['b_reliable'].mean():.1%}")
w(f"depth 10 km : {fixed_depth_share:.2%}")

w()
w("== AUDIT PENYARINGAN ==")
w(audit.to_string(index=False))

w()
w("== NILAI KEDALAMAN TERSERING ==")
w(df["depth"].value_counts().head(10).to_string())

w()
w("== MAGNITUDE TYPE ==")
_mt = [c for c in df_raw.columns if c.lower() in ("magnitude type", "magnitude_type")]
w(df_raw[_mt[0]].value_counts(dropna=False).head(12).to_string() if _mt
  else "(kolom tidak ada)")

w()
w("== BOBOT KANAL ==")
for n, we, wh in zip(CHANNELS, META["w_equal"], META["w_entropy"]):
    w(f"{n:8s} equal={we:.4f} entropy={wh:.4f} std={h3_summary[n].std():.4f} "
      f"mean={h3_summary[n].mean():.4f}")

w()
w("== KORELASI KANAL ==")
w(h3_summary[CHANNELS].corr().round(3).to_string())

w()
w("== SHI ==")
w(f"range equal   : [{h3_summary['shi'].min():.4f}, {h3_summary['shi'].max():.4f}]")
w(f"range entropy : [{h3_summary['shi_entropy'].min():.4f}, {h3_summary['shi_entropy'].max():.4f}]")
w(f"mean / median : {h3_summary['shi'].mean():.4f} / {h3_summary['shi'].median():.4f}")
w(f"spearman rho  : {rho:.4f}")
w(h3_summary["hazard_level"].value_counts().reindex(TIER_LABELS, fill_value=0).to_string())
w("distribusi kuartil:")
w(h3_summary["hazard_quartile"].value_counts().sort_index().to_string())

w()
w("== PEMICU RISIKO DOMINAN ==")
w(h3_summary["risk_driver"].value_counts().to_string())

w()
w("== DISTRIBUSI b TERUKUR ==")
_b = h3_summary.loc[h3_summary["b_estimated"], "b_value"]
w(_b.describe().round(4).to_string())
w(f"sel di batas bawah {B_MIN} : {(_b <= B_MIN).sum():,}")
w(f"sel di batas atas {B_MAX}  : {(_b >= B_MAX).sum():,}")

w()
w("== TOP 20 ==")
w(top20_out[["h3_index", "province", "location_type", "total_events", "n_mc",
             "max_magnitude", "default_depth_share", "b_value_imputed",
             "b_estimated", "b_reliable", "shi", "hazard_level",
             "risk_driver"]].round(4).to_string(index=False))

w()
w("== SENSITIVITAS N_min ==")
w(nmin_table.to_string(index=False))

w()
w("== SENSITIVITAS RESOLUSI ==")
w(res_table.to_string(index=False))

w()
w("== SENSITIVITAS KEDALAMAN DEFAULT ==")
w(dd_table.to_string(index=False))
w(f"mean s_norm   : {h3_summary['s_norm'].mean():.4f} -> {s_dd['s_norm'].mean():.4f}")
w(f"mean shi      : {h3_summary['shi'].mean():.4f} -> {s_dd['shi'].mean():.4f}")
w(f"overlap top20 : {len(top20_base & top20_dd)}/20")
w()
w(dd_check.round(4).to_string(index=False))

w()
w("== UJI KEWAJARAN: EVENT TERBESAR DI KATALOG ==")
w(f"magnitudo maksimum katalog : {df['magnitude'].max():.2f}  (plafon m_norm {M_CEILING})")
w(df.nlargest(10, "magnitude")[big_cols].to_string(index=False))

w()
w("== UJI KEWAJARAN: EVENT RUJUKAN ==")
w(f"ambang bawah Top-20 (SHI) : {top20_floor:.4f}")
w("rata-rata nasional : " + "  ".join(f"{c}={nat[c]:.4f}" for c in CHANNELS))
w(ref_table.to_string(index=False))

w()
w("== UJI BATAS LINTANG (rupture Aceh 2004) ==")
w(f"koridor {KORIDOR_BUJUR[0]}-{KORIDOR_BUJUR[1]} BT, >= 1960 : {len(kor):,} event")
w(tabel.to_string())
w(f"di atas {LINTANG_POTONG} LU, terbuang : {len(atas):,} ({len(atas)/len(kor):.1%})")
w(f"sekuens 2004-2005 : {len(seq):,} event, terbuang {len(atas_seq):,} "
  f"({len(atas_seq)/len(seq):.1%})" if len(seq) else "sekuens 2004-2005: tidak ada")
if len(seq):
    w(f"lintang maksimum sekuens : {seq['latitude'].max():.2f} LU")

w()
w("== NILAI b PER TIPE MAGNITUDO ==")
w(type_table.to_string(index=False))

w()
w("== TIPE DOMINAN PER SEL ==")
w(sel["tipe_dominan"].value_counts().to_string())
w(f"porsi tipe dominan: median {sel['porsi_dominan'].median():.2%}")
w("b menurut tipe dominan sel:")
w(terukur.groupby("tipe_dominan")["b_value"].agg(["count", "mean", "std"]).round(4).to_string())

w()
w("== HOMOGENISASI MAGNITUDO ==")
w(f"event dikonversi : {df_hom['mag_dikonversi'].mean():.1%}")
w(f"mag maksimum     : {df['magnitude'].max():.2f} -> {df_hom['magnitude'].max():.2f}")
w("b per tipe, sebelum dan sesudah:")
w(banding.to_string(index=False))
w(hom_table.to_string(index=False))
w(f"b nasional     : {META['national_b']:.4f} -> {m_hom['national_b']:.4f}")
w(f"mean SHI       : {h3_summary['shi'].mean():.4f} -> {s_hom['shi'].mean():.4f}")
w("pemisahan efek, lewat tumpang-tindih Top-20:")
w(f"  asli@{MC:.1f} vs asli@{MC_HOM:.2f}            : {len(top20_base & top20_mc)}/20  (ambang saja)")
w(f"  asli@{MC_HOM:.2f} vs terhomogenisasi@{MC_HOM:.2f} : {len(top20_mc & top20_hom)}/20  (konversi saja)")
w(f"  asli@{MC:.1f} vs terhomogenisasi@{MC_HOM:.2f}  : {len(top20_base & top20_hom)}/20  (keduanya)")
w(f"Spearman rho   : {rho_hom:.4f}")
w(f"subset Mw saja : {len(df_mw):,} event ({len(df_mw)/len(df):.2%})")

w()
w("== FILE OUTPUT ==")
for f in (OUT_GEOJSON, OUT_MAP, out("top_20_critical_hexagons.csv"),
          out("sensitivity_nmin.csv"), out("sensitivity_resolution.csv"),
          out("sensitivity_default_depth.csv"), out("validation_reference_events.csv"),
          out("sensitivity_magnitude_homogenisation.csv")):
    w(f"{f:<46} {os.path.getsize(f)/1e6:.2f} MB" if os.path.exists(f)
      else f"{f:<46} TIDAK ADA")

text = "\n".join(lines)
with open(REVIEW_PATH, "w", encoding="utf-8") as fh:
    fh.write(text)

print(f"Ditulis ke {REVIEW_PATH}  ({len(text):,} karakter)\n")
print(text)

---
## Limitasi yang harus dibawa ke bagian Diskusi

1. **Katalog tanpa declustering.** Swarm gempa susulan kaya akan event kecil, yang
   menaikkan nilai *b* lokal dan karenanya *menurunkan* $B_{\text{inv}}$. Wilayah dengan
   mega-sekuens seperti Aceh 2004 bisa jadi skor kanal tegangannya lebih rendah daripada
   yang disiratkan tatanan tektoniknya. Ini bacaan yang benar atas *b* sebagai kuantitas
   fisis, dan sekaligus limitasi pemakaiannya sebagai proksi bahaya di sini. Declustering
   Gardner-Knopoff atau Reasenberg masuk ke pekerjaan lanjutan.
2. **Perancu kerapatan jaringan.** Membatasi hitungan ke $M \ge M_c$ mengurangi, tetapi
   tidak menghilangkan, celah deteksi Indonesia timur. Periksa apakah pita SHI rendah di
   sana mengikuti cakupan stasiun, bukan seismisitas.
3. **Kanal yang berkorelasi.** $M_{\max}$ adalah statistik nilai ekstrem dan naik seiring
   ukuran sampel, jadi ia berkorelasi dengan $N_{\text{norm}}$. Bobot sama rata
   menghitung ganda aktivitas sejauh itu; bobot entropi adalah mitigasi yang dinyatakan,
   dan Bagian 7 mengukur selisihnya.
4. **Kedalaman default.** Proporsi event tepat di 10 km, yang dilaporkan di Bagian 2,
   membatasi seberapa besar bagian kanal kedalaman dangkal yang benar-benar nyata.
5. **Nilai *b* hasil imputasi.** Sel yang membawa $b_{\text{national}}$ diberi flag di
   seluruh pipeline. Nyatakan berapa bagian Top-20 yang bertumpu pada nilai imputasi.
6. **$M_c$ dibuat konstan secara spasial.** Satu $M_c$ nasional untuk kepulauan dengan
   cakupan stasiun yang tidak merata adalah penyederhanaan. $M_c$ yang bervariasi secara
   spasial masuk ke pekerjaan lanjutan.
7. **Tidak ada lapisan keterpaparan.** Indeks ini memeringkat bahaya, bukan risiko.
   Populasi, stok bangunan, dan respons situs semuanya belum ada.

> Label tier (`Low`, `Moderate`, `High`, `Very High`) sengaja dibiarkan berbahasa
> Inggris: nilai-nilai itu tertulis ke GeoJSON, dibaca folium, dan diuji oleh assertion
> di Bagian 6, sekaligus menjaga konsistensi dengan tabel klasifikasi risiko di PRD
> Section 3.3.